# T135 checkpoint inference with Q10

User-authorized run of the upstream Q10 notebook for the completed T135 final900 checkpoint, raw and EMA weights. Q08 is disabled. Training and checkpoint bytes remain unchanged; outputs stay in the original T135 run folder.

# Q10: test-time BatchNorm adaptation and class-balanced prediction (evaluation only)

NYU Tandon Deep Learning (Fall 2026), Project 1 ([Kaggle competition](https://www.kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl)).
Checklist task **T136** (owner J.H.). The course staff explicitly allowed two prediction-time techniques, which must
be **disclosed in the write-up**:

1. **Test-time BatchNorm adaptation**: recompute the BatchNorm means and variances from the 2,000 unlabeled test images
   and predict with them.
2. **Class-balanced prediction**: rescale the per-class probabilities over all 2,000 test images so that every class
   receives an equal share (the test set is assumed balanced, 200 per class), then take the argmax.

The hidden test images are shifted against the training data (brighter, higher contrast: clean validation ~98% versus
public ~0.90; `进展记录.md` §0b, [Q08](Q08_shiftval.ipynb)). This notebook trains nothing. It loads saved E checkpoints,
rebuilds every model from the architecture saved with it, and evaluates every BN variant × plain/flip × balancing on

| Domain | Images | BN statistics adapted on |
|---|---|---|
| clean validation | the training notebooks' 5k split (`SPLIT_SEED=0`) | the 5,000 validation images themselves |
| Q08 shifted validation | each set of the locked [Q08](Q08_shiftval.ipynb) proxy (default: all 13 corruptions at severity 1; last checkpoint) | that set's 5,000 images |
| test | the 2,000 competition test images | the 2,000 test images |

| BN variant | Running statistics | File tag |
|---|---|---|
| `source` | the checkpoint's saved buffers for these weights (EMA buffers for EMA) | `bnsrc` (test: `-bal` / `-balx` CSVs only) |
| `full` | recomputed from the domain's images: BN layers reset, `momentum=None` (cumulative average), eval transform | `bnfull` |
| `mixNNN` | `(1 - a) * source + a * full`, means and variances separately, a = NNN/100 (0.25 / 0.5 / 0.75; Q08: 0.5) | `bnmixNNN` |

| Balancing | Rule |
|---|---|
| `none` | argmax of the softmax probabilities |
| `sinkhorn` (`-bal`) | Sinkhorn iterative proportional fitting: class weights w such that every class receives n/k of the probability mass (rows keep mass 1); label = argmax p_ic w_c |
| `exact` (`-balx`) | exactly n/k images per class with the largest total log-probability (linear sum assignment) |

- **Statistics pass.** On a deep copy of the model: `model.eval()`, then only the BN layers `train()` with
  `momentum=None` after `reset_running_stats()`. ShakeDrop (multiplies by 1 − p in eval), DropPath (identity) and the
  network stay in evaluation mode; no gradients; parameters are checked bit-identical afterwards. Default: statistics
  from the **un-flipped** images (`BN_STATS_INCLUDE_FLIP=False`); plain and flip are then predicted with the same
  statistics. The source model is never modified (checked after every domain and every checkpoint).
- **Baseline.** The `source` variant must reproduce the stored per-image probabilities of the same checkpoint epoch and
  weights (`*-validation.npz`, test `*_probabilities.npz`) within `BASELINE_PROB_ATOL` and
  `BASELINE_MAX_LABEL_CHANGES`; otherwise no test file is written for that checkpoint.
- **Precision.** fp32 without autocast, contiguous tensors, batches of 1,000: the path (`load_model` +
  `predict_logits`) of every recorded headline.
- **Selection.** None. Every variant is reported; validation and Q08 are diagnostics, not hidden-test estimates.
- **Full-data refits** (full-data extension 1 of version 1, added 2026-11-05 before any full-data checkpoint was
  evaluated; every rule above for 45k/5k checkpoints is unchanged). A checkpoint trained on all 50,000 images
  (`VAL_SIZE=0`, e.g. T131) is recognized from what it saved: `val_size` 0 and an empty `val_idx`, with `train_idx`
  required to be all 50,000 indices (the cache fingerprint check is unchanged). It has only a last checkpoint and
  gets the **test domain only**: clean validation and Q08 are skipped with the reason printed and recorded (the model
  was trained on those 5,000 images, and the Q08 sets are made from them). Its baseline is the stored test probability
  exports of the same checkpoint and weights (same tolerances; with none stored, no test file is written). Test files
  and names are those of any checkpoint; the per-checkpoint JSON has no `.npz`, and its validation/Q08 table columns
  are NaN.

**Usage on Colab.** Upload this notebook, select an **A100** (or L4) GPU, keep the competition data/`cache.zip` in
`MyDrive/DL_Project1/` as for training, check `Q10_RUNS` (section 1) and *Run all*; approve the Drive mount. Runs or
checkpoints that are not on this account's Drive (or the read-only `Q10_EXTRA_RESULT_ROOTS`) are skipped with a
printed reason, so the same notebook runs unchanged on either teammate's Drive. Expected time on an A100 with the
defaults (estimated from the Q08/Q05 throughputs, not yet measured for Q10): about 1 min for data and the Q08 set, then
per bottleneck-110 run ~3 min for the last checkpoint (with Q08) and ~0.5–1 min for the best checkpoint; T106
(basic-110) about half of that, T123 (bottleneck-164) about 1.5×. About 15 min for T128, T129, T122 and T106, about
20 min with T123; an L4 is roughly 3× slower. The notebook prints the forward counts before it starts.

**Outputs** (under `RESULT_DIR` only; nothing is overwritten): per checkpoint and weights
`result/evaluations/validation/<RUN_ID>/<MM-DD-HH-mm>_<id>_q10-bnadapt-v1-<role>-ep<E>-<w>.json/.npz` (all metrics;
float16 validation probabilities and the predicted labels of every variant, view and balancing on validation and Q08)
and, in `result/evaluations/test/<RUN_ID>/`, all four kinds of submission CSV per view (`<view>` plain / flip; file
names `<MM-DD-HH-mm>_<id>_q10-<variant tag>-<role>-ep<E>-<w>-<view>...`):

| Kind | Files |
|---|---|
| original | the stored training-time export of the checkpoint (`*_submission.csv` with its `*_probabilities.npz`); **not rewritten**: the `source` variant reproduces it (baseline check) and the test report lists it |
| balancing only | `q10-bnsrc-...-bal_submission.csv` (Sinkhorn) and `...-balx_submission.csv` (exact) |
| BN only | per BN-adapted variant (`bnfull`, `bnmixNNN`): `..._submission.csv` with its `..._probabilities.npz` (also holding the Sinkhorn probabilities and every label set) |
| BN + balancing | per BN-adapted variant: `...-bal_submission.csv` and `...-balx_submission.csv` |

With the defaults that is 28 CSVs and 8 `.npz` per checkpoint and weights (printed before the run), plus one
`..._q10-bnadapt-v1-<role>-ep<E>-<w>_test-report.json`; shared `result/reports/<MM-DD-HH-mm>_Q10_bn-adapt-v1_summary.csv`
and `..._report.json`. The CSVs are **never submitted** by this notebook; every Kaggle submission needs the user's
approval. Balanced CSVs for already exported probabilities: `Archive/tools/balance_submissions.py`. Its 2026-11-04
CSVs (`11-04-20-55_*-bal[x]_submission.csv`) duplicate this notebook's `bnsrc` CSVs: once these outputs are synced into
the repository, `Archive/tools/prune_balanced_duplicates.py` (dry run unless `--delete`) removes every one of them that
Q10 reproduced (a CSV linked in `checklist.md`, e.g. a submitted one, only if its labels are identical, re-pointing
the link) and keeps the rest (raw weights, runs not evaluated here).

CPU checks: `Archive/tests/test_q10_balance.py` (balancing, the tool, this notebook's structure),
`Archive/tests/test_q10_bn_adapt.py` (end to end on stand-in checkpoints, including full-data refits) and the self-test
in section 8 (BN code on stand-in models, on the runtime's device). Not yet run on a GPU; no Q10 result exists yet.

## 1. Configuration

Edit this cell, then Run all. `Q10_RUNS` lists Drive run IDs (registry names are translated with `Q10_DRIVE_NAMES`).
Every listed run is evaluated at its saved best and last checkpoints (`Q10_CHECKPOINTS`) with EMA weights
(`Q10_WEIGHTS`; add `"model"` for raw weights). Checkpoints and their stored artifacts are searched in `RESULT_DIR`
first, then in each read-only `Q10_EXTRA_RESULT_ROOTS` folder; a run or checkpoint found in no root is skipped with the
reason printed, problems with identity, split, integrity or I/O are recorded as failures and the next run follows.

`VAL_SIZE`, `SPLIT_SEED`, `Q10_COMPETITION_CHECKS` and `Q08_LOCKED_SET_SHA256` must stay as given. The BN variants,
the balancing rules and the reproduction check are fixed in sections 6–8 (version 1, 2026-11-04, before any Q10
result was seen). Full-data refits (`VAL_SIZE=0` checkpoints such as T131) can be listed like any run: they are
recognized from the checkpoint and evaluated on the test images only (full-data extension 1, 2026-11-05, section 7);
nothing needs to be set.

In [17]:
# Q10 / checklist T136: test-time BatchNorm adaptation and class-balanced prediction. Evaluation only, no training.
# Both techniques were explicitly allowed by the course staff for prediction and must be disclosed in the write-up.
# Edit the values below, then Run all. Nothing here submits to Kaggle (every submission needs the user's approval).

# Runs: exact Drive run IDs (the folder names under result/checkpoints/). Registry (canonical) names are translated
# with Q10_DRIVE_NAMES. Each run is rebuilt from the architecture saved in its checkpoint. A run or checkpoint found
# in no result root is skipped with a printed reason, so this list works unchanged on either teammate's Drive.
# Full-data refits (VAL_SIZE=0, e.g. T131) may be listed too: recognized from the checkpoint, test images only.
Q10_RUNS = ["11-04-17-39_E_pyramid-ver-41-T135-d218full-ls01"]  # User-authorized T135 checkpoint inference
# Registry (canonical) name -> Drive folder name (Archive/run_identity_map.json); Drive keeps the original names.
Q10_DRIVE_NAMES = {
    "11-04-01-58_E_pyramid-ver-27-T128-asam-t122reg": "11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg",
    "11-03-02-04_E_pyramid-ver-24-T122-replica-d110": "11-03-02-04_E_pyramid-ver-20-T122-replica-d110",
    "11-03-02-38_E_pyramid-ver-25-T123-replica-d164": "11-03-02-38_E_pyramid-ver-24-T123-replica-d164",
    "11-02-03-08_E_pyramid-ver-12-T106-trivialaug-drop": "11-02-03-08_E_pyramid-ver-10-T106-trivialaug-drop",
}
Q10_CHECKPOINTS = ("last",)  # User-authorized T135 checkpoint inference
Q10_WEIGHTS = ("model", "ema")  # User-authorized T135 checkpoint inference
                                    # for raw weights (about doubles the time)
# Read-only result folders of other accounts, searched after RESULT_DIR (as Q05/Q08): H.W.'s shared DL_Project1 folder
# added to MyDrive as the shortcut HW_DL_Project1. On H.W.'s Colab it is simply absent (warning, then skipped); a
# shortcut to J.H.'s folder could be added here the same way, e.g. "/content/drive/MyDrive/JH_DL_Project1/result".
Q10_EXTRA_RESULT_ROOTS = []  # User-authorized T135 checkpoint inference
Q10_EXTRA_ROOT_WAIT_S = 30          # Colab only: a Drive shortcut can appear a little after mounting; wait this long

# BatchNorm adaptation (section 7). Variants: "source" (saved statistics), "full" (statistics recomputed from the
# evaluated images: cumulative average, momentum=None, eval transform, only BN layers in training mode) and
# "mixNNN" = (1 - a) * source + a * target for every a below (means and variances blended separately).
BN_BLEND_ALPHAS = (0.25, 0.5, 0.75)  # clean validation and test
Q08_BLEND_ALPHAS = (0.5,)            # Q08 sets (each extra alpha costs 13 x 10,000 forwards per checkpoint)
BN_STATS_INCLUDE_FLIP = False        # True: the statistics pass also sees every image flipped (2x pass). Default:
                                     # statistics from the un-flipped images only; plain and flip are predicted with them
BN_STATS_BATCH_SIZE = 1000           # statistics batches: the images split into equal parts of at most this size

# Q08 shifted validation (section 5): regenerated exactly as Q08_shiftval.ipynb and used only if the whole-set
# fingerprint equals the locked value. Each evaluated set is its own domain (BN adapted on that set's 5,000 images).
Q08_EVAL = False  # User-authorized T135 checkpoint inference
Q08_LOCKED_SET_SHA256 = "3f12d750b28139569b8912c175dce67dfaadaa471afc838d9f4af62538b797d1"  # Q08 lock (2026-10-31)
Q08_EVAL_SEVERITIES = (1,)           # severity 1 of every corruption: the mildest, closest to the measured test shift
Q08_EVAL_CORRUPTIONS = None          # None = all 13 corruptions; or a list of Q08 corruption names
Q08_CHECKPOINTS = ("last",)          # Q08 for these checkpoint roles only (Q08's pre-declared primary is the last
                                     # checkpoint); ("best", "last") adds ~2.5 min per bottleneck-110 run on an A100

# Class-balanced prediction (section 6): Sinkhorn balancing on every domain; the exact equal-count assignment also on:
BALANCE_EXACT_DOMAINS = ("val", "test")  # add "q08" for the Q08 sets (about 0.5 s per set, variant and view)

# Baseline check: the "source" variant must reproduce the stored per-image probabilities of the same checkpoint
# epoch and weights (validation exports and test probability exports) within these tolerances (cuDNN/TF32 and batch
# size differences between runtimes stay below ~1.5e-3 and 1 label, measured by Q05); otherwise no test files are
# written for that checkpoint.
BASELINE_PROB_ATOL = 5e-3
BASELINE_MAX_LABEL_CHANGES = 3

Q10_WRITE_TEST_OUTPUTS = True    # per BN-adapted variant and view: probabilities .npz, unbalanced / Sinkhorn / exact
                                 # CSVs; source variant: Sinkhorn / exact CSVs only (its probabilities and unbalanced
                                 # CSV are the stored training-time export, which it must reproduce)
Q10_SELF_TEST = True             # stand-in checks of the BN code before any checkpoint is read (seconds)
Q10_ALLOW_SHA256_MISMATCH = False  # a checkpoint whose SHA-256 differs from its recorded value is skipped unless True
Q10_STOP_ON_ERROR = True  # User-authorized T135 checkpoint inference
Q10_SMOKE_N = 0                  # > 0: first N validation images only, no Q08 and no test files (marked -smoke)

# Fixed split and data identity: keep these values (checked against the training notebooks' split; a full-data
# checkpoint's training indices are checked against all 50,000 images instead)
VAL_SIZE = 5000
SPLIT_SEED = 0
Q10_COMPETITION_CHECKS = True    # require the competition cache fingerprint and the known split hashes

# Hardware (None = automatic). Prediction is the training notebooks' path: fp32 without autocast, contiguous
# tensors, batches of 1,000 (PyTorch's default TF32 settings are left unchanged).
DEVICE = None            # "cuda" | "mps" | "cpu"; None = $CIFAR_DEVICE, else cuda > mps > cpu
AMP = "off"              # not used; kept for the shared environment cell
CHANNELS_LAST = False
CUDNN_BENCHMARK = None
PIN_MEMORY = None
WORKERS = None
EVAL_BATCH_SIZE = 1000

# Runtime host, independent of the editor: auto | colab | kaggle | local.
# For a VS Code kernel connected to Colab, set "colab" if auto detection is ambiguous.
RUNTIME_ENV = "auto"     # explicit value overrides CIFAR_ENV; auto uses CIFAR_ENV then detection

# Paths (None = $CIFAR_DATA_DIR / $CIFAR_CACHE_DIR / $CIFAR_RESULT_DIR, else the default of the environment)
AUTO_DOWNLOAD = False      # Kaggle download/login code is commented out; use uploaded data
SAVE_CACHE_TO_DRIVE = False  # inference only: reuse MyDrive/DL_Project1/cache.zip, never rewrite it
DATA_DIR = None            # Colab: MyDrive/DL_Project1/data/ (folder or override with a ZIP path)
CACHE_DIR = None
RESULT_DIR = None          # checkpoints are read from RESULT_DIR/checkpoints (then the extra roots); outputs go here
SHOW_SUMMARY = False       # the copied model cell prints no layer table here

# Baseline switch values for the verbatim model cell's own baseline build (section 4). Every checkpoint is rebuilt
# from its saved architecture options, whatever these values are.
DROP_PATH = 0.0
SHAKEDROP = 0.0
BLOCK = "basic"
DEPTH = 110
ALPHA = 98

## 2. Environment, paths and device

The two code cells below are verbatim copies of `E_pyramid_t106_shakedrop.ipynb`'s environment cells (the tests check
this), so detection, Drive mounting and paths behave exactly as in training: Colab reads and writes
`MyDrive/DL_Project1/result` and keeps the cache in `/content/cache`; local runs use `project1/data`, `project1/cache`
and `project1/result`. On Colab select an A100 or L4 GPU.

In [18]:
import copy
import csv
import hashlib
import importlib.util
import importlib.metadata
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import tempfile
import zlib
import zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.ticker import MaxNLocator
from PIL import Image

ENVS = ("local", "colab", "kaggle")
KAGGLE_SLUG = "cifar-10-res-net-architecture-challenge-dl"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DL_Project1"
WORK_DIRS = {"colab": "/content", "kaggle": "/kaggle/working"}
COLAB_CKPT_EVERY = 5  # Drive writes are slow; best-ckpt.pt and the final epoch are always saved
PROJECT_DIR = Path.cwd().resolve()
LOCAL_PROJECT_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == "Archive" else PROJECT_DIR
# Start method for the AutoAugment DataLoader workers: "fork" lets them use the Dataset class defined in this
# notebook. Under "spawn" (macOS, Windows) they cannot import it, so the pipeline runs in the main process there.
MP_CONTEXT = "fork" if sys.platform.startswith("linux") else None

_warned = set()


def _getenv(name):
    value = os.environ.get(name, "").strip()
    return value or None


def warn_once(msg):
    if msg not in _warned:
        _warned.add(msg)
        print(f"Warning: {msg}", flush=True)


def _module_exists(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def detect_env():
    """Detect the kernel host, independent of the notebook editor (browser or VS Code)."""
    configured = str(globals().get("RUNTIME_ENV") or "auto").strip().lower()
    if configured not in ("auto",) + ENVS:
        raise ValueError(f"RUNTIME_ENV={configured!r}; expected auto or one of {ENVS}")
    requested = configured if configured != "auto" else (_getenv("CIFAR_ENV") or "auto").lower()
    if requested not in ("auto",) + ENVS:
        raise ValueError(f"CIFAR_ENV={requested!r}; expected auto or one of {ENVS}")
    if requested != "auto":
        return requested
    # Runtime markers beat directory names: /kaggle/input may also exist on Colab.
    if any(_getenv(key) for key in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
        return "colab"
    if _getenv("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if _module_exists("google.colab"):
        return "colab"
    # A directory alone is weak evidence; use the full layout only as a fallback.
    if os.path.isdir("/kaggle/input") and os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"


def work_dir(env=None):
    env = env or detect_env()
    return Path(_getenv("CIFAR_WORK_DIR") or WORK_DIRS.get(env, PROJECT_DIR))


def drive_dir():
    return Path(_getenv("CIFAR_DRIVE_DIR") or DRIVE_PROJECT_DIR)


def drive_mounted():
    return drive_dir().parent.is_dir()  # .../MyDrive exists once Drive is mounted


def kaggle_input_dir():
    return Path(_getenv("CIFAR_KAGGLE_INPUT") or "/kaggle/input")


def find_competition_data(root=None, max_depth=6):
    """Directories under `root` that contain train_labels.csv, best match first.

    Breadth-first, follows symlinks, skips *_images folders (50k files) and hidden folders.
    """
    root = Path(root or kaggle_input_dir())
    found, frontier = [], [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError:
            continue
        if any(e.name == "train_labels.csv" and e.is_file() for e in entries):
            found.append(d)
        if depth < max_depth:
            frontier += [(Path(e.path), depth + 1) for e in entries
                         if e.is_dir() and not e.name.endswith("_images") and not e.name.startswith(".")]

    def rank(d):
        complete = (d / "test.csv").is_file() and (d / "sample_submission.csv").is_file()
        return (not complete, KAGGLE_SLUG not in str(d), len(d.parts), str(d))

    return sorted(found, key=rank)


@dataclass
class Paths:
    env: str
    data_dir: Path
    cache_dir: Path
    result_dir: Path


def resolve_paths(data_dir=None, cache_dir=None, result_dir=None, env=None):
    """Data, cache and runs directories: argument > environment variable > environment default."""
    env = env or detect_env()
    data = data_dir or _getenv("CIFAR_DATA_DIR")
    cache = cache_dir or _getenv("CIFAR_CACHE_DIR")
    runs = result_dir or _getenv("CIFAR_RESULT_DIR")
    if env == "local":
        data = data or LOCAL_PROJECT_DIR / "data"
        cache = cache or LOCAL_PROJECT_DIR / "cache"
        runs = runs or LOCAL_PROJECT_DIR / "result"
    elif env == "colab":
        cache = cache or work_dir(env) / "cache"
        data = data or drive_dir() / "data"  # manually uploaded ZIPs or extracted competition files
        if runs is None:
            if drive_mounted():
                runs = drive_dir() / "result"
            else:
                runs = work_dir(env) / "result"
                warn_once(f"Google Drive is not mounted ({drive_dir().parent} missing); writing results to "
                          f"{runs}, which is lost when the runtime is recycled")
    else:  # kaggle
        cache = cache or work_dir(env) / "cache"
        runs = runs or work_dir(env) / "result"
        if data is None:
            found = find_competition_data()
            if found:
                data = found[0]
            else:
                data = work_dir(env) / "data"
                warn_once(f"no train_labels.csv found under {kaggle_input_dir()}; checking cache or manually supplied data")
    return Paths(env, Path(data), Path(cache), Path(runs))


def default_ckpt_every(env=None):
    return COLAB_CKPT_EVERY if (env or detect_env()) == "colab" else 1


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:  # macOS
        return os.cpu_count() or 1


def default_workers(device_type):
    """DataLoader workers (used by the per-image AutoAugment pipeline only)."""
    n = available_cpus()
    if device_type == "cuda":
        return min(8, max(2, n - 1))
    return min(4, max(1, n // 2))  # 4 on the 12-core M4 Pro, as before


def get_device(name=None):
    """`name` (DEVICE), else $CIFAR_DEVICE, else cuda > mps > cpu."""
    name = name or _getenv("CIFAR_DEVICE") or "auto"
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(name)


def cuda_bf16_native(device):
    return torch.cuda.get_device_capability(device) >= (8, 0)  # Ampere or newer (L4, A100, ...)


def resolve_amp(requested, device):
    """'bf16', 'fp16' or 'off'. requested: auto | on | off | bf16 | fp16 (bools accepted)."""
    req = {True: "on", False: "off", None: "auto"}.get(requested, requested)
    req = str(req).lower()
    if req == "off":
        return "off"
    if device.type != "cuda":
        if req != "auto":
            # MPS: fp16 autocast gave no speed-up on an M4 Pro, and torch 2.5.1 GradScaler fails on MPS.
            print(f"Warning: AMP is only used on CUDA; training in fp32 on {device.type}.")
        return "off"
    if req == "fp16":
        return "fp16"
    if cuda_bf16_native(device):
        return "bf16"
    if req == "bf16":
        print("Warning: this GPU has no native bf16; using fp16 autocast + GradScaler instead.")
    return "fp16"


# torch.compile modes for the training step (README "Compilation"); a speed option, never part of the recipe.
COMPILE_MODES = ("off", "default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs")


def resolve_compile(requested, device, name="COMPILE"):
    """One of COMPILE_MODES. Compilation is used on CUDA only; elsewhere it warns and runs uncompiled ("off")."""
    if not isinstance(requested, str) or requested not in COMPILE_MODES:
        raise ValueError(f"{name} = {requested!r}; choose one of {COMPILE_MODES}")
    if requested != "off" and device.type != "cuda":
        print(f"Warning: {name}={requested!r} is only used on CUDA; running uncompiled on {device.type}.")
        return "off"
    return requested


def resolve_hardware(args, device):
    """Fill in the automatic (None) speed settings on `args` for `device`, in place."""
    cuda = device.type == "cuda"
    args.amp = resolve_amp(getattr(args, "amp", "auto"), device)
    args.compile = resolve_compile(getattr(args, "compile", "off"), device)
    if args.channels_last is None:
        args.channels_last = cuda
    elif args.channels_last and device.type == "mps":
        # torch 2.5.1 MPS: backward through conv/BN/ReLU in channels_last raises a stride error.
        print("Warning: CHANNELS_LAST is not supported on MPS with this PyTorch; disabling it.")
        args.channels_last = False
    if getattr(args, "cudnn_benchmark", None) is None:
        args.cudnn_benchmark = cuda
    if getattr(args, "pin_memory", None) is None:
        args.pin_memory = cuda
    if getattr(args, "workers", None) is None:
        args.workers = default_workers(device.type)
    if MP_CONTEXT is None:
        args.workers = 0  # see MP_CONTEXT
    return args


def device_name(device):
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        return f"cuda ({props.name}, {props.total_memory / 2**30:.0f} GB)"
    if device.type == "cpu":
        return f"cpu ({available_cpus()} cores)"
    return f"{device.type} ({platform.machine()})"


def _on(x):
    return "on" if x else "off"


def summary_line(paths, device=None, args=None):
    """One line: environment, device, torch, speed settings and the output directories."""
    import torch

    parts = [f"env {paths.env}"]
    if device is not None:
        parts.append(device_name(device))
    parts.append(f"torch {torch.__version__}")
    if args is not None:
        parts.append(f"amp {args.amp} | channels_last {_on(args.channels_last)} | "
                     f"cudnn.benchmark {_on(args.cudnn_benchmark)} | workers {args.workers} | "
                     f"pin_memory {_on(args.pin_memory)}")
        if getattr(args, "compile", "off") != "off":
            parts.append(f"compile {args.compile}")
    parts.append(f"data {paths.data_dir} | cache {paths.cache_dir} | result {paths.result_dir}")
    return "[env] " + " | ".join(parts)

def run_timestamp():
    return datetime.now(ZoneInfo("America/New_York")).strftime("%m-%d-%H-%M")


# Exact historical aliases: checkpoint bytes and signed export manifests remain unchanged.
RUN_ID_ALIASES = {'10-30-08-40_A_resnet-ver-1-cutout-ema': '10-30-08-40_A_resnet-ver-1-T00-cutout-ema', '10-30-09-44_B_wrn-ver-1-cutout-ema': '10-30-09-44_B_wrn-ver-1-T00-cutout-ema', '10-30-09-46_C_seresnet-ver-1-cutout-ema': '10-30-09-46_C_seresnet-ver-1-T00-cutout-ema', '10-30-09-50_C_seresnet-ver-2-cutout-ema': '10-30-09-50_C_seresnet-ver-2-T00-cutout-ema', '10-30-19-41_B_wrn-ver-2-wide-ema': '10-30-19-41_B_wrn-ver-2-T08-wide-ema', '10-30-20-03_A_resnet-ver-6-eca-ema': '10-30-20-03_A_resnet-ver-2-T48-eca-ema', '10-30-20-04_C_seresnet-ver-8-se8-ema': '10-30-20-04_C_seresnet-ver-3-T06-se8-ema', '10-30-21-05_E_pyramid-ver-1-basic110-ema': '10-30-21-05_E_pyramid-ver-1-T18-basic110-ema', '10-30-21-07_G_resnext-ver-1-grouped-ema': '10-30-21-07_G_resnext-ver-1-T42-grouped-ema', '10-30-21-17_A_resnet-ver-7-mixup-ema': '10-30-21-17_A_resnet-ver-3-T49-mixup-ema', '10-30-21-18_B_wrn-ver-8-mixup-ema': '10-30-21-18_B_wrn-ver-3-T50-mixup-ema', '10-30-21-42_F_shake-ver-1-shake40': '10-30-21-42_F_shake-ver-1-T35-shake40', '10-30-21-47_C_seresnet-ver-31-mixup-ema': '10-30-21-47_C_seresnet-ver-4-T51-mixup-ema', '10-30-23-09_A_resnet-ver-4-early-ema': '10-30-23-09_A_resnet-ver-4-T22-early-ema', '10-30-23-10_B_wrn-ver-6-deep-ema': '10-30-23-10_B_wrn-ver-4-T24-deep-ema', '10-30-23-43_C_seresnet-ver-3-autoaug-cutout': '10-30-23-43_C_seresnet-ver-5-T01-autoaug-cutout', '10-30-23-56_A_resnet-ver-5-mid-ema': '10-30-23-56_A_resnet-ver-5-T26-mid-ema', '10-31-00-42_B_wrn-ver-5-balanced-ema': '10-31-00-42_B_wrn-ver-5-T23-balanced-ema', '10-31-00-50_C_seresnet-ver-19-reflect-ema': '10-31-00-50_C_seresnet-ver-6-T31-reflect-ema', '10-30-22-34_F_shake-ver-2-shake40-1800ep': '10-30-22-34_F_shake-ver-2-T35-shake40-1800ep', '10-31-20-06_E_pyramid-ver-3-T61-autoaug-only': '10-31-20-06_E_pyramid-ver-3-T69-autoaug-only'}


def canonical_run_id(run_id):
    return RUN_ID_ALIASES.get(run_id, run_id)


def run_task_id(run_id):
    match = re.search(r"-ver-\d+-(T\d{2,})-", canonical_run_id(run_id))
    return match.group(1) if match else "T00"


def run_paths(result_root, run_id, create=False):
    """One timestamped ID across directories grouped by artifact type."""
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Run ID must be a single directory name")
    run_id = canonical_run_id(run_id)
    root = Path(result_root)
    folders = {"checkpoints": "checkpoints", "configs": "configs", "logs": "logs",
               "validation": "evaluations/validation", "test": "evaluations/test",
               "plots": "plots", "progress": "progress", "manifests": "manifests", "exports": "exports"}
    paths = SimpleNamespace(root=root, run_id=run_id,
                            **{key: root / folder / run_id for key, folder in folders.items()})
    if not paths.checkpoints.exists():
        for legacy, canonical in RUN_ID_ALIASES.items():
            old_dir = root / "checkpoints" / legacy
            if canonical == run_id and old_dir.is_dir():
                paths.checkpoints = old_dir
                break
    if create:
        for key in folders:
            getattr(paths, key).mkdir(parents=True, exist_ok=True)
    return paths


def paths_for_run(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if checkpoint_dir.parent.name != "checkpoints":
        raise ValueError("Expected result/checkpoints/<RUN_ID>; legacy flat runs must be migrated explicitly")
    return run_paths(checkpoint_dir.parent.parent, checkpoint_dir.name)


def artifact_path(artifacts, category, role, extension, event=False, unique=False):
    """Stable training paths; dated evaluation/export paths with collision-safe role suffixes."""
    folder = getattr(artifacts, category)
    prefix = artifacts.run_id
    if event:
        identifier = artifacts.run_id.split('_', 1)[1] if '_' in artifacts.run_id else artifacts.run_id
        prefix = f"{run_timestamp()}_{identifier}"
    stem = f"{prefix}_{role}"
    path = folder / (stem + extension)
    if unique:
        index = 1
        while any(folder.glob(path.stem + '*')):
            index += 1
            path = folder / (f"{stem}-{index}" + extension)
    return path


def checkpoint_path(run_dir, which):
    if which not in ('best', 'last'):
        raise ValueError("Checkpoint role must be best or last")
    run_dir = Path(run_dir)
    canonical = canonical_run_id(run_dir.name)
    candidates = [run_dir.parent / canonical / f"{canonical}_{which}-ckpt.pt",
                  run_dir / f"{run_dir.name}_{which}-ckpt.pt"]
    for legacy, target in RUN_ID_ALIASES.items():
        if target == canonical:
            candidates.extend([run_dir.parent / canonical / f"{legacy}_{which}-ckpt.pt",
                               run_dir.parent / legacy / f"{legacy}_{which}-ckpt.pt"])
    return next((p for p in candidates if p.is_file()), candidates[0])


In [19]:
ENV = detect_env()
print(f"[runtime] detected {ENV} | kernel {sys.executable} | cwd {Path.cwd()} | "
      f"RUNTIME_ENV={RUNTIME_ENV!r} | CIFAR_ENV={_getenv('CIFAR_ENV')!r}")
if ENV == "colab":
    try:
        from google.colab import drive
    except ImportError:  # CIFAR_ENV=colab outside Google Colab (testing)
        print("google.colab is not available: skipping drive.mount")
    else:
        drive.mount("/content/drive")

PATHS = resolve_paths(DATA_DIR, CACHE_DIR, RESULT_DIR, ENV)
DEV = get_device(DEVICE)
if ENV == "colab" and DEV.type != "cuda" and (DEVICE or _getenv("CIFAR_DEVICE")) != "cpu":
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > A100 GPU, then Runtime > Run all again")
HW = resolve_hardware(SimpleNamespace(amp=AMP, channels_last=CHANNELS_LAST, cudnn_benchmark=CUDNN_BENCHMARK,
                                      pin_memory=PIN_MEMORY, workers=WORKERS), DEV)
print(summary_line(PATHS, DEV, HW))
print(f"python {platform.python_version()} | torchvision {torchvision.__version__} | numpy {np.__version__} | "
      f"{platform.platform()} | {available_cpus()} CPUs")
if ENV == "colab":
    print(f"Google Drive folder {drive_dir()}: {'found' if drive_dir().is_dir() else 'NOT FOUND'}")
if ENV == "kaggle":
    print(f"Kaggle input {kaggle_input_dir()}: train_labels.csv in "
          f"{[str(d) for d in find_competition_data()] or 'none'}")
if DEV.type == "cuda" and shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

[runtime] detected colab | kernel /usr/bin/python3 | cwd /content | RUNTIME_ENV='auto' | CIFAR_ENV=None
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[env] env colab | cuda (NVIDIA A100-SXM4-40GB, 39 GB) | torch 2.11.0+cu130 | amp off | channels_last off | cudnn.benchmark on | workers 8 | pin_memory on | data /content/drive/MyDrive/DL_Project1/data | cache /content/cache | result /content/drive/MyDrive/DL_Project1/result
python 3.13.16 | torchvision 0.26.0+cu130 | numpy 2.4.6 | Linux-6.6.159+-x86_64-with-glibc2.39 | 12 CPUs
Google Drive folder /content/drive/MyDrive/DL_Project1: found
Fri Nov  6 04:23:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name      

## 3. Data, the shared validation split and the test images

The data helpers and the data cell are verbatim copies of the training notebook's cells (a verified local cache, then
a compatible Drive `cache.zip`, then the uploaded data); the next cell is Q05's verbatim copy of the split,
normalization, inference and checkpoint helpers of `E_pyramid_t106_shakedrop.ipynb`. The split cell recomputes the
45k/5k split with `stratified_split`, checks the cache fingerprint (`42749a486a13…`) and the SHA-256 of the validation
indices and labels shared by every saved validation export, and loads the 2,000 test images with their
sample-submission order. Each checkpoint's saved `train_idx` and `val_idx` must equal this split exactly.

In [20]:
NUM_CLASSES = 10
CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
CACHE_FORMAT = 2
SOURCE_CSVS = ("train_labels.csv", "test.csv", "sample_submission.csv")
CACHE_ARRAYS = ("train_images.npy", "train_labels.npy", "train_ids.npy", "test_images.npy", "test_ids.npy")


def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _decode_pngs(paths):
    def load(p):
        with Image.open(p) as im:
            return np.asarray(im.convert("RGB"), dtype=np.uint8)

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        return np.stack(list(pool.map(load, paths, chunksize=256)))


def _save_npy(path, arr):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def _copy_csvs(data_dir, cache_dir):
    if data_dir.resolve() == cache_dir.resolve():
        return
    for name in SOURCE_CSVS:
        tmp = cache_dir / (name + ".tmp")
        shutil.copyfile(data_dir / name, tmp)
        os.replace(tmp, cache_dir / name)


def _write_meta(cache_dir, n_train, n_test, shape, note):
    files = {name: {"sha256": sha256_file(cache_dir / name), "bytes": (cache_dir / name).stat().st_size}
             for name in CACHE_ARRAYS + SOURCE_CSVS}
    fingerprint = hashlib.sha256("".join(f"{n} {files[n]['sha256']}\n" for n in sorted(files)).encode())
    meta = {"format": CACHE_FORMAT, "n_train": n_train, "n_test": n_test, "shape": list(shape),
            "fingerprint": fingerprint.hexdigest(), "files": files, "note": note,
            "written": datetime.now().isoformat(timespec="seconds")}
    tmp = cache_dir / "meta.json.tmp"
    tmp.write_text(json.dumps(meta, indent=2))
    os.replace(tmp, cache_dir / "meta.json")  # written last: a cache without it is incomplete
    return meta


def read_meta(cache_dir):
    path = Path(cache_dir) / "meta.json"
    return json.loads(path.read_text()) if path.exists() else None


def build_cache(data_dir, cache_dir):
    """Decode all PNGs once into uint8 arrays (N, 32, 32, 3) and save them with the CSVs and meta.json."""
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(data_dir / "train_labels.csv")
    print(f"[cache] decoding {len(train):,} training PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "train_images" / f for f in train["filename"]])
    _save_npy(cache_dir / "train_images.npy", x)
    _save_npy(cache_dir / "train_labels.npy", train["label"].to_numpy(np.int64))
    _save_npy(cache_dir / "train_ids.npy", train["id"].to_numpy(np.int64))

    test = pd.read_csv(data_dir / "test.csv")
    print(f"[cache] decoding {len(test):,} test PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "test_images" / f for f in test["filename"]])
    _save_npy(cache_dir / "test_images.npy", x)
    _save_npy(cache_dir / "test_ids.npy", test["id"].to_numpy(np.int64))

    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), x.shape[1:], f"built from {data_dir}")
    print(f"[cache] wrote {cache_dir} (fingerprint {meta['fingerprint'][:12]})", flush=True)
    return meta


def _upgrade_format1(data_dir, cache_dir):
    """Add CSV copies and hashes to a format-1 cache if its arrays match the CSVs. Returns success."""
    train = pd.read_csv(data_dir / "train_labels.csv")
    test = pd.read_csv(data_dir / "test.csv")
    try:
        x_train = np.load(cache_dir / "train_images.npy", mmap_mode="r")
        x_test = np.load(cache_dir / "test_images.npy", mmap_mode="r")
        ok = (x_train.shape[0] == len(train) and x_test.shape[0] == len(test)
              and x_train.shape[1:] == x_test.shape[1:]
              and np.array_equal(np.load(cache_dir / "train_labels.npy"), train["label"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "train_ids.npy"), train["id"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "test_ids.npy"), test["id"].to_numpy(np.int64)))
    except (OSError, ValueError):
        ok = False
    if not ok:
        return False
    shape = x_train.shape[1:]
    del x_train, x_test
    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), shape, f"format 1 cache upgraded with {data_dir}")
    print(f"[cache] upgraded {cache_dir} to format {CACHE_FORMAT} (arrays unchanged, "
          f"fingerprint {meta['fingerprint'][:12]})", flush=True)
    return True


def _cache_problem(meta, cache_dir, data_dir, has_csvs):
    """None if the format-2 cache is complete and matches the CSVs in data_dir, else the reason."""
    for name, info in meta["files"].items():
        path = cache_dir / name
        if not path.exists() or path.stat().st_size != info["bytes"]:
            return f"{name} is missing or has the wrong size"
    if has_csvs and data_dir.resolve() != cache_dir.resolve():
        for name in SOURCE_CSVS:
            if sha256_file(data_dir / name) != meta["files"][name]["sha256"]:
                return f"{data_dir / name} differs from the cached copy"
    return None


def ensure_cache(data_dir, cache_dir):
    """Return the cache dir, building, upgrading or rebuilding the cache only when needed.

    A complete format-2 cache is used as is; if data_dir also holds the CSVs, they must match the
    cached copies (otherwise the cache is rebuilt from the PNGs).
    """
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    has_csvs = all((data_dir / n).is_file() for n in SOURCE_CSVS)
    has_pngs = has_csvs and (data_dir / "train_images").is_dir() and (data_dir / "test_images").is_dir()
    meta = read_meta(cache_dir)
    if meta is not None and meta.get("format") == CACHE_FORMAT:
        problem = _cache_problem(meta, cache_dir, data_dir, has_csvs)
        if problem is None:
            return cache_dir
        print(f"[cache] {problem}; rebuilding", flush=True)
    elif meta is not None and has_csvs and all((cache_dir / n).exists() for n in CACHE_ARRAYS):
        if _upgrade_format1(data_dir, cache_dir):
            return cache_dir
        print("[cache] format 1 cache does not match the CSVs; rebuilding", flush=True)
    if not has_pngs and meta is not None and meta.get("format") is None:
        raise FileNotFoundError(
            f"{cache_dir} is a format-1 cache and {data_dir} has no CSVs to upgrade it with; "
            f"set DATA_DIR to the folder with the competition CSVs")
    if not has_pngs:
        raise FileNotFoundError(
            f"no usable cache in {cache_dir} and no PNG images in {data_dir} to build one; set DATA_DIR "
            f"(folder with train_labels.csv, train_images/, ...) or CACHE_DIR, or on Colab upload cache.zip "
            f"to {drive_dir()}")
    build_cache(data_dir, cache_dir)
    return cache_dir


def verify_cache(cache_dir):
    """Check every cached file against the sha256 in meta.json; return meta (raises on a mismatch)."""
    cache_dir = Path(cache_dir)
    meta = read_meta(cache_dir)
    _check_cache_manifest(meta)
    bad = [n for n, info in meta["files"].items()
           if not (cache_dir / n).exists() or sha256_file(cache_dir / n) != info["sha256"]]
    if bad:
        raise RuntimeError(f"cache files do not match meta.json: {bad}")
    n_train, n_test = validate_cache_arrays(cache_dir)
    if (meta["n_train"], meta["n_test"], meta["shape"]) != (n_train, n_test, [32, 32, 3]):
        raise RuntimeError("Cache metadata does not match its arrays")
    return meta


def validate_cache_arrays(cache_dir):
    """Validate shapes, labels, IDs, and the correspondence between arrays and CSVs."""
    cache_dir = Path(cache_dir)
    frames = {name: pd.read_csv(cache_dir / name) for name in SOURCE_CSVS}
    for name, columns in (("train_labels.csv", ("id", "filename", "label")),
                          ("test.csv", ("id", "filename")),
                          ("sample_submission.csv", ("id", "label"))):
        frame = frames[name]
        if not set(columns).issubset(frame.columns) or frame.empty:
            raise ValueError(f"{name}: missing columns or no rows")
        if not pd.api.types.is_integer_dtype(frame["id"]) or not frame["id"].is_unique:
            raise ValueError(f"{name}: IDs must be unique integers")
        if "filename" in frame and (frame["filename"].isna().any() or not frame["filename"].is_unique):
            raise ValueError(f"{name}: filenames must be present and unique")
    train, test = frames["train_labels.csv"], frames["test.csv"]
    if not pd.api.types.is_integer_dtype(train["label"]) or not train["label"].between(0, 9).all():
        raise ValueError("Training labels must be integers in 0..9")
    for split, frame in (("train", train), ("test", test)):
        images = np.load(cache_dir / f"{split}_images.npy", mmap_mode="r", allow_pickle=False)
        ids = np.load(cache_dir / f"{split}_ids.npy", allow_pickle=False)
        if images.dtype != np.uint8 or images.shape != (len(frame), 32, 32, 3):
            raise ValueError(f"{split} images must be uint8 NHWC with shape (N,32,32,3)")
        if not np.array_equal(ids, frame["id"].to_numpy()):
            raise ValueError(f"{split} array IDs differ from CSV order")
    labels = np.load(cache_dir / "train_labels.npy", allow_pickle=False)
    if not np.issubdtype(labels.dtype, np.integer) or not np.array_equal(labels, train["label"].to_numpy()):
        raise ValueError("Cached labels differ from training CSV")
    if set(frames["sample_submission.csv"]["id"]) != set(test["id"]):
        raise ValueError("Sample submission and test IDs differ")
    return len(train), len(test)


def cache_fingerprint(cache_dir):
    meta = read_meta(cache_dir) or {}
    return meta.get("fingerprint")


def load_train(data_dir, cache_dir, mmap=False):
    cache_dir = ensure_cache(data_dir, cache_dir)
    images = np.load(cache_dir / "train_images.npy", mmap_mode="r" if mmap else None)
    return images, np.load(cache_dir / "train_labels.npy"), np.load(cache_dir / "train_ids.npy")


def load_test(data_dir, cache_dir):
    cache_dir = ensure_cache(data_dir, cache_dir)
    return np.load(cache_dir / "test_images.npy"), np.load(cache_dir / "test_ids.npy")


def _cache_matches_source(meta, source_dir):
    """An explicitly supplied set of CSVs must agree with a reused cache."""
    source_dir = Path(source_dir)
    return all(not (source_dir / name).is_file() or
               sha256_file(source_dir / name) == meta['files'][name]['sha256'] for name in SOURCE_CSVS)


def _check_cache_manifest(meta):
    required = set(CACHE_ARRAYS + SOURCE_CSVS)
    if (not isinstance(meta, dict) or meta.get('format') != CACHE_FORMAT
            or not isinstance(meta.get('files'), dict) or set(meta['files']) != required):
        raise ValueError('Cache needs a format-2 manifest with all required files')
    for info in meta['files'].values():
        if (not isinstance(info, dict) or not isinstance(info.get('bytes'), int)
                or info['bytes'] <= 0 or not isinstance(info.get('sha256'), str)
                or not re.fullmatch(r'[0-9a-f]{64}', info['sha256'])):
            raise ValueError('Invalid cache file size or SHA256 in manifest')
    fingerprint = hashlib.sha256(''.join(
        f"{name} {meta['files'][name]['sha256']}\n" for name in sorted(required)).encode()).hexdigest()
    if fingerprint != meta.get('fingerprint'):
        raise ValueError('Cache ZIP fingerprint differs from its manifest')
    return required


def verify_cache_zip(path):
    """Check ZIP contents and every uncompressed file hash, not just the ZIP header."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or set(names) != set(CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',)):
            raise ValueError('Cache ZIP must contain exactly the required files at its root')
        meta = json.loads(z.read('meta.json'))
        for name in _check_cache_manifest(meta):
            info = meta['files'][name]
            digest = hashlib.sha256()
            with z.open(name) as f:
                for block in iter(lambda: f.read(1 << 22), b''):
                    digest.update(block)
            if z.getinfo(name).file_size != info['bytes'] or digest.hexdigest() != info['sha256']:
                raise ValueError(f'Cache ZIP checksum mismatch: {name}')
    return meta


def extract_zip_checked(archive, destination):
    """Extract inside a staging folder, rejecting escaping paths and symlink entries."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if ('\\' in member.filename or not target.is_relative_to(root)
                    or (member.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        z.extractall(destination)  # decompression also checks each member's CRC


def publish_data_directory(staged, destination):
    """Keep the previous directory until the validated replacement can be published."""
    staged, destination = Path(staged), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    previous = destination.with_name(f'.{destination.name}.previous-{time.time_ns()}')
    if destination.exists():
        os.replace(destination, previous)
    try:
        os.replace(staged, destination)
    except BaseException:
        if previous.exists():
            try:
                os.replace(previous, destination)
            except OSError:
                # Keep recovery files even if the filesystem also refuses rollback.
                print(f'[data] previous files preserved at {previous}; restore them before retrying')
        raise
    else:
        if previous.is_dir() and not previous.is_symlink():
            shutil.rmtree(previous)
        else:
            previous.unlink(missing_ok=True)


def drive_cache_candidates(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    pattern = re.compile(r'cache(?: \(\d+\))?\.zip$')
    return sorted((p for p in folder.iterdir() if p.is_file() and pattern.fullmatch(p.name)),
                  key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True)


def fetch_drive_cache(folder, cache_dir, source_dir=None):
    """Try Drive cache copies newest first; never replace a cache with an unverified ZIP."""
    cache_dir = Path(cache_dir)
    cache_dir.parent.mkdir(parents=True, exist_ok=True)
    for archive in drive_cache_candidates(folder):
        try:
            with tempfile.TemporaryDirectory(prefix='.restore-cache-', dir=cache_dir.parent) as tmp:
                local_zip, staged = Path(tmp) / 'cache.zip', Path(tmp) / 'cache'
                shutil.copyfile(archive, local_zip)
                meta = verify_cache_zip(local_zip)
                if source_dir is not None and not _cache_matches_source(meta, source_dir):
                    print(f'[cache] skipping {archive.name}: source CSVs differ')
                    continue
                extract_zip_checked(local_zip, staged)
                verify_cache(staged)
                publish_data_directory(staged, cache_dir)
            print(f'[cache] restored {archive} -> {cache_dir}')
            return cache_dir
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError) as exc:
            print(f'[cache] cannot reuse {archive.name}: {type(exc).__name__}; trying another source')
    raise FileNotFoundError('No valid compatible Drive cache was found')


def save_drive_cache(cache_dir, folder):
    """Publish a verified cache.zip once; skip writing an identical valid backup."""
    cache_dir, folder = Path(cache_dir), Path(folder)
    meta = verify_cache(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / 'cache.zip'
    if destination.is_file():
        try:
            if verify_cache_zip(destination)['fingerprint'] == meta['fingerprint']:
                print(f'[cache] Drive backup already verified: {destination}')
                return destination
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError):
            print('[cache] replacing an invalid Drive backup after verifying its replacement')
    partial = folder / f'.cache-{time.time_ns()}.zip.tmp'
    try:
        with tempfile.TemporaryDirectory(prefix='.export-cache-', dir=cache_dir.parent) as tmp:
            archive = Path(tmp) / 'cache.zip'
            with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
                for name in CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',):
                    z.write(cache_dir / name, arcname=name)
            verify_cache_zip(archive)
            shutil.copyfile(archive, partial)
            verify_cache_zip(partial)  # check the Drive copy before replacing the old backup
            os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    print(f'[cache] saved reusable Drive backup: {destination}')
    return destination


def find_raw_data(root, require_images=False):
    """Find CSVs under a wrapper folder, optionally checking every referenced PNG exists."""
    for folder in find_competition_data(root):
        if not all((folder / name).is_file() for name in SOURCE_CSVS):
            continue
        if not require_images:
            return folder
        complete = True
        for split, csv_name in (('train', 'train_labels.csv'), ('test', 'test.csv')):
            frame = pd.read_csv(folder / csv_name)
            if 'filename' not in frame or frame.empty or frame['filename'].isna().any():
                complete = False
                break
            image_root = (folder / f'{split}_images').resolve()
            for filename in frame['filename']:
                image = (image_root / str(filename)).resolve()
                if not image.is_relative_to(image_root) or not image.is_file():
                    complete = False
                    break
            if not complete:
                break
        if complete:
            return folder
    return None


# Kaggle download and login disabled: use manually uploaded data instead.
# def run_kaggle_download():
#     """Use KaggleHub in this notebook process and honor its returned cache/input path."""
#     if not _getenv('KAGGLE_CONFIG_DIR') and (Path('/content') / 'kaggle.json').is_file():
#         os.environ['KAGGLE_CONFIG_DIR'] = '/content'
#     try:
#         kagglehub = importlib.import_module('kagglehub')
#     except ModuleNotFoundError as exc:
#         if exc.name != 'kagglehub':
#             raise RuntimeError('A KaggleHub dependency is missing. Run %pip install -U "kagglehub>=1,<2", '
#                                'restart the kernel, and rerun from the configuration cell.') from None
#         print('[download] installing KaggleHub in this kernel environment')
#         installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
#                                     '--disable-pip-version-check', 'kagglehub>=1,<2'],
#                                    capture_output=True, text=True)
#         if installed.returncode:
#             raise RuntimeError('Cannot install KaggleHub. Check internet access and use Python 3.11+; '
#                                'alternatively provide extracted data or a saved cache.')
#         importlib.invalidate_caches()
#         kagglehub = importlib.import_module('kagglehub')
#     print(f'[download] KaggleHub competition_download({KAGGLE_SLUG!r})', flush=True)
#     try:
#         # Do not assume a ZIP filename or pass newer optional API arguments.
#         # This also preserves credentials set by kagglehub.login() in this kernel.
#         downloaded = Path(kagglehub.competition_download(KAGGLE_SLUG))
#     except Exception as exc:
#         response = getattr(exc, 'response', None)
#         status = getattr(response, 'status_code', None)
#         if not isinstance(status, int):
#             status = getattr(exc, 'error_code', None)
#         status_text = f', HTTP {status}' if isinstance(status, int) and 100 <= status <= 599 else ''
#         kind = type(exc).__name__
#         if status == 401 or kind in ('UnauthenticatedError', 'CredentialError'):
#             print('[auth] Kaggle login is needed for this download. Enter your own API token in the login prompt.')
#             try:
#                 kagglehub.login()
#             except Exception as login_error:
#                 raise RuntimeError(
#                     f'Cannot open/complete Kaggle login ({type(login_error).__name__}). '
#                     'For notebook widgets, install ipywidgets and enable widget support in your editor, '
#                     'or use a masked getpass prompt to set KAGGLE_API_TOKEN (see README.md). '
#                     'Then rerun this data cell.') from None
#             # Notebook login displays an asynchronous widget. The kernel must become idle
#             # for its Login button to work; do not block/poll or download again immediately.
#             raise RuntimeError(
#                 f'Data setup paused for Kaggle login ({kind}{status_text}). '
#                 'Complete the login prompt above, then rerun THIS DATA CELL and continue the remaining cells. '
#                 'Do not restart the kernel after login. Existing data and valid caches need no login.') from None
#         if status == 403:
#             advice = ('Your account may need competition access or rule acceptance. '
#                       'If you need to switch credentials, use kagglehub.login() or set KAGGLE_API_TOKEN. '
#                       f'Confirm competition access and accepted rules at https://www.kaggle.com/competitions/{KAGGLE_SLUG}/rules.')
#         elif status == 404:
#             advice = f'Check the competition name {KAGGLE_SLUG!r} and that your account can access it.'
#         elif status == 429:
#             advice = 'Kaggle rate-limited the request. Wait before retrying.'
#         elif kind in ('ConnectionError', 'ConnectTimeout', 'ReadTimeout', 'Timeout'):
#             advice = 'Check the runtime internet connection and retry the data cell.'
#         elif isinstance(exc, OSError):
#             advice = 'Check local free space and write permissions for the KaggleHub cache.'
#         else:
#             advice = ('Check KaggleHub installation and account access; use kagglehub.login() or '
#                       'KAGGLE_API_TOKEN if authentication is required.')
#         # Keep useful diagnostics without echoing request URLs/tokens from exception text.
#         raise RuntimeError(f'KaggleHub download failed ({kind}{status_text}). {advice} '
#                            'Rerun the data cell after correcting the issue.') from None
#     if not downloaded.exists():
#         raise FileNotFoundError(f'KaggleHub returned a path that does not exist: {downloaded}')
#     print(f'[download] KaggleHub returned: {downloaded}')
#     return downloaded
#
#
# def download_competition_data(data_dir):
#     """Normalize a KaggleHub folder/ZIP or a legacy downloaded ZIP into the raw data directory."""
#     data_dir = Path(data_dir)
#     data_dir.parent.mkdir(parents=True, exist_ok=True)
#     legacy_archive = data_dir.parent / 'downloads' / KAGGLE_SLUG / f'{KAGGLE_SLUG}.zip'
#     downloaded = None
#     if legacy_archive.is_file():
#         try:
#             with zipfile.ZipFile(legacy_archive) as z:
#                 if z.testzip() is None:
#                     downloaded = legacy_archive
#         except (OSError, zipfile.BadZipFile, zlib.error, EOFError):
#             pass
#     if downloaded is None:
#         downloaded = run_kaggle_download()
#     else:
#         print(f'[download] reusing completed legacy archive: {downloaded}')
#     if downloaded.is_dir() and (downloaded.resolve().is_relative_to(data_dir.resolve())
#                                 or data_dir.resolve().is_relative_to(downloaded.resolve())):
#         raw = find_raw_data(downloaded, require_images=True)
#         if raw is not None:
#             return raw  # reuse overlapping complete data without moving or modifying the provider cache
#         raise ValueError('KaggleHub cache overlaps DATA_DIR. Choose a separate DATA_DIR for extraction.')
#     with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
#         extracted = Path(tmp) / 'extracted'
#         if downloaded.is_dir():
#             # KaggleHub normally already extracted the competition archive. Work on a copy:
#             # its cache (or Kaggle's read-only input mount) must not be moved or unpacked in place.
#             shutil.copytree(downloaded, extracted, copy_function=shutil.copyfile)
#             for directory, _, _ in os.walk(extracted):
#                 Path(directory).chmod(0o700)  # only the working copy must allow nested ZIP extraction
#         elif zipfile.is_zipfile(downloaded):
#             extract_zip_checked(downloaded, extracted)
#         else:
#             raise ValueError(f'Expected a competition folder or ZIP from KaggleHub, received {downloaded}')
#         unpacked = set()
#         for _ in range(4):
#             raw = find_raw_data(extracted, require_images=True)
#             if raw is not None:
#                 publish_data_directory(raw, data_dir)
#                 print(f'[data] ready: {data_dir}')
#                 return data_dir
#             nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
#             if not nested:
#                 break
#             for path in nested:
#                 target = path.parent
#                 if path.stem in ('train_images', 'test_images'):
#                     with zipfile.ZipFile(path) as z:
#                         if not any(name.startswith(path.stem + '/') for name in z.namelist()):
#                             target = target / path.stem
#                 extract_zip_checked(path, target)
#                 unpacked.add(path)
#     raise ValueError(f'Downloaded files at {downloaded} lack complete train_images/, test_images/ and '
#                      'the three CSVs. Inspect that folder or set DATA_DIR to complete extracted data. '
#                      'If the KaggleHub cache is incomplete, refresh it with '
#                      'kagglehub.competition_download(..., force_download=True).')
#
#

def prepare_uploaded_data(uploaded, data_dir):
    """Read a user-supplied folder/ZIP; unpack only into local staging, without network access."""
    uploaded, data_dir = Path(uploaded), Path(data_dir)
    if not uploaded.exists():
        raise FileNotFoundError(
            f'No complete data or valid cache. Upload the competition ZIP or extracted files to '
            f'{uploaded}. On Colab the default is MyDrive/DL_Project1/data/. '
            'Kaggle downloads and login are disabled.')
    if (uploaded.resolve().is_relative_to(data_dir.resolve())
            or data_dir.resolve().is_relative_to(uploaded.resolve())):
        raise ValueError('Uploaded source and local extraction directory must be separate.')
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'[data] reading manually uploaded files: {uploaded}')
    with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
        extracted = Path(tmp) / 'extracted'
        if uploaded.is_dir():
            shutil.copytree(uploaded, extracted, copy_function=shutil.copyfile)
            for directory, _, _ in os.walk(extracted):
                Path(directory).chmod(0o700)
        elif zipfile.is_zipfile(uploaded):
            extract_zip_checked(uploaded, extracted)
        else:
            raise ValueError(f'Expected an uploaded competition folder or ZIP, received {uploaded}')
        unpacked = set()
        for _ in range(4):
            raw = find_raw_data(extracted, require_images=True)
            if raw is not None:
                publish_data_directory(raw, data_dir)
                print(f'[data] ready: {data_dir}')
                return data_dir
            nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
            if not nested:
                break
            for path in nested:
                target = path.parent
                if path.stem in ('train_images', 'test_images'):
                    with zipfile.ZipFile(path) as z:
                        if not any(name.startswith(path.stem + '/') for name in z.namelist()):
                            target = target / path.stem
                extract_zip_checked(path, target)
                unpacked.add(path)
    raise ValueError(
        f'Uploaded files at {uploaded} lack complete train_images/, test_images/, '
        'train_labels.csv, test.csv and sample_submission.csv. Upload all competition files; '
        'ZIP and extracted-folder layouts are supported.')


def prepare_data(paths, auto_download=False, save_to_drive=True):
    """Valid local cache -> compatible Drive cache -> manually uploaded folder/ZIP."""
    if auto_download:
        raise ValueError("Kaggle download/login is disabled. Keep AUTO_DOWNLOAD=False and upload data manually.")
    cache = paths.cache_dir
    persist = paths.env == 'colab' and save_to_drive
    if persist and not drive_mounted():
        raise RuntimeError('Mount Google Drive before data setup to save cache.zip, or explicitly set '
                           'SAVE_CACHE_TO_DRIVE=False for temporary storage only.')
    source = find_raw_data(paths.data_dir) or paths.data_dir
    meta = None
    if (cache / 'meta.json').is_file():
        try:
            candidate = verify_cache(cache)
            if _cache_matches_source(candidate, source):
                meta = candidate
                print(f'[cache] reusing verified local cache: {cache}')
            else:
                print('[cache] source CSVs changed; rebuilding from the requested data')
        except (OSError, ValueError, KeyError, RuntimeError, EOFError) as exc:
            print(f'[cache] local cache is incomplete or invalid ({type(exc).__name__})')
    if meta is None and paths.env == 'colab' and drive_mounted():
        try:
            fetch_drive_cache(drive_dir(), cache, source)
            meta = verify_cache(cache)
        except FileNotFoundError:
            pass
    if meta is None:
        if paths.data_dir.resolve() == cache.resolve():
            raise ValueError('Use separate DATA_DIR and CACHE_DIR when building from uploaded data')
        raw = find_raw_data(paths.data_dir, require_images=True)
        if raw is None:
            raw = prepare_uploaded_data(paths.data_dir, cache.parent / 'data_extracted')
        paths.data_dir = raw
        cache.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.build-cache-', dir=cache.parent) as tmp:
            staged = Path(tmp) / 'cache'
            build_cache(raw, staged)
            meta = verify_cache(staged)
            publish_data_directory(staged, cache)
    else:
        paths.data_dir = source
    if persist:
        save_drive_cache(cache, drive_dir())
    return cache, meta


In [21]:
CACHE, META = prepare_data(PATHS, auto_download=AUTO_DOWNLOAD, save_to_drive=SAVE_CACHE_TO_DRIVE)
# Keep later training/evaluation cells on the resolved source folder (including wrapper directories).
DATA_DIR = str(PATHS.data_dir)
CACHE_DIR = str(CACHE)
print(f"[cache] {CACHE}: format {META['format']} | {META['n_train']:,} train / {META['n_test']:,} test images "
      f"{tuple(META['shape'])} | fingerprint {META['fingerprint'][:12]} | sha256 verified")
counts = np.bincount(np.load(CACHE / "train_labels.npy"), minlength=NUM_CLASSES)
print("Training images per class: " + ", ".join(f"{c} {n:,}" for c, n in zip(CLASSES, counts)))

[cache] reusing verified local cache: /content/cache
[cache] /content/cache: format 2 | 50,000 train / 2,000 test images (32, 32, 3) | fingerprint 42749a486a13 | sha256 verified
Training images per class: airplane 5,000, automobile 5,000, bird 5,000, cat 5,000, deer 5,000, dog 5,000, frog 5,000, horse 5,000, ship 5,000, truck 5,000


In [22]:
# Verbatim copies of E_pyramid_t106_shakedrop.ipynb helpers (split, normalization statistics, crop padding,
# inference, checkpoint identity, submission check). Do not edit: tests/test_q05_shift_tta.py compares every
# definition below with that notebook. model_fingerprint, validate_checkpoint and load_model use section 4's
# model code (MODEL, build_model, ...).
def _per_class(n, what):
    if not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0:
        raise ValueError(f"{what} must be a nonnegative integer, got {n!r}")
    if n % NUM_CLASSES:
        raise ValueError(f"{what} must be a multiple of {NUM_CLASSES} for a class-balanced split, got {n}")
    return n // NUM_CLASSES


def stratified_split(labels, val_size, seed=0):
    """Hold out `val_size` images, the same number from every class. Returns sorted (train_idx, val_idx)."""
    labels = np.asarray(labels)
    if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or not len(labels):
        raise ValueError("labels must be a nonempty one-dimensional integer array")
    if np.any((labels < 0) | (labels >= NUM_CLASSES)):
        raise ValueError("labels must be in 0..9")
    k = _per_class(val_size, "val_size")
    if val_size >= len(labels):
        raise ValueError("Validation size must leave at least one training image")
    if np.any(np.bincount(labels, minlength=NUM_CLASSES) < k):
        raise ValueError("Not enough examples per class for the requested validation split")
    all_idx = np.arange(len(labels))
    if val_size == 0:
        return all_idx, np.array([], dtype=np.int64)
    k = _per_class(val_size, "val_size")
    rng = np.random.default_rng(seed)
    val = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:k] for c in range(NUM_CLASSES)])
    val = np.sort(val)
    return np.setdiff1d(all_idx, val), val


def channel_stats(images):
    """Per-channel mean/std in [0, 1] units over uint8 NHWC images (computed in chunks)."""
    s = np.zeros(3)
    ss = np.zeros(3)
    n = 0
    for i in range(0, len(images), 5000):
        chunk = np.asarray(images[i:i + 5000], dtype=np.float64) / 255.0
        s += chunk.sum(axis=(0, 1, 2))
        ss += (chunk ** 2).sum(axis=(0, 1, 2))
        n += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]
    mean = s / n
    std = np.sqrt(ss / n - mean ** 2)
    return mean.tolist(), std.tolist()


def to_float_nchw(x_u8, mean, std, channels_last=False):
    """uint8 NHWC on device -> normalized float NCHW (channels_last strides are free from NHWC)."""
    x = x_u8.permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last if channels_last else torch.contiguous_format)


def crop_pad(x_u8, pad=4, padding_mode="constant"):
    """Pad uint8 NHWC images before random cropping; reflect excludes the edge pixel (same code as scheme C/T31)."""
    if padding_mode not in ("constant", "reflect"):
        raise ValueError("crop_padding_mode must be constant or reflect")
    if padding_mode == "constant":
        return F.pad(x_u8, (0, 0, pad, pad, pad, pad))  # original black-padding path
    h, w = x_u8.shape[1:3]
    if not isinstance(pad, int) or not 0 <= pad < min(h, w):
        raise ValueError("Reflection padding must be nonnegative and smaller than the image")
    # Integer indexing preserves uint8 on both CPU and CUDA, with no random draws.
    rows = torch.arange(-pad, h + pad, device=x_u8.device)
    cols = torch.arange(-pad, w + pad, device=x_u8.device)
    rows = torch.where(rows < 0, -rows, torch.where(rows >= h, 2 * h - 2 - rows, rows))
    cols = torch.where(cols < 0, -cols, torch.where(cols >= w, 2 * w - 2 - cols, cols))
    return x_u8.index_select(1, rows).index_select(2, cols)


def replace_file(src, dst):
    """Publish a completed file by rename; never overwrite a good checkpoint by partial copy."""
    try:
        os.replace(src, dst)
    except OSError as exc:
        raise OSError(f"Could not atomically publish {dst}; the previous file is preserved. "
                      "Use a filesystem that supports same-directory rename.") from exc


def write_bytes_atomic(data, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    replace_file(tmp, path)


def load_checkpoint(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(state, dict) or "args" not in state:
        return state
    original = state["args"].get("run_name")
    canonical = canonical_run_id(original)
    if original != canonical:
        state.setdefault("original_identity", {"run_id": original,
                         "model_version": state["args"].get("model_version")})
        state["args"] = dict(state["args"], run_name=canonical,
                             model_version=int(re.search(r"-ver-(\d+)-", canonical).group(1)),
                             task_id=run_task_id(canonical))
    return state


@torch.inference_mode()
def predict_logits(model, images_u8, mean, std, device, batch_size=1000, flip=False, channels_last=False):
    """Logits for uint8 NHWC images (numpy or tensor); returns a CPU float tensor.

    With flip=True, returns the logits of horizontally flipped images instead.
    """
    model.eval()
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(images_u8), batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + batch_size])).to(device)
        x = to_float_nchw(xb, mean_t, std_t, channels_last)
        if flip:
            x = x.flip(3)
        out.append(model(x).float().cpu())
    return torch.cat(out)


def predict_probs(model, images_u8, mean, std, device, batch_size=1000, tta=True, channels_last=False):
    """Softmax probabilities; with tta=True, averages the original and horizontally flipped view."""
    kw = dict(batch_size=batch_size, channels_last=channels_last)
    probs = F.softmax(predict_logits(model, images_u8, mean, std, device, **kw), 1)
    if tta:
        probs = 0.5 * (probs + F.softmax(predict_logits(model, images_u8, mean, std, device, flip=True, **kw), 1))
    return probs


CHECKPOINT_FORMAT = 3


def write_json_atomic(value, path):
    write_bytes_atomic((json.dumps(value, indent=2) + "\n").encode(), path)


def model_fingerprint(model):
    options = getattr(model, "_project_architecture", ARCH_DEFAULTS)
    definition = getattr(model, "_project_description", MODEL_DESCRIPTION) + "\n" + repr(model)
    # Preserve the exact legacy baseline hash. New variants also bind options and implementation version.
    if options != ARCH_DEFAULTS:
        definition += "\n" + json.dumps(options, sort_keys=True)
    return hashlib.sha256(definition.encode()).hexdigest()


def validate_checkpoint(ckpt, fingerprint, expected_model_hash=None, run_id=None):
    if ckpt.get("format") != CHECKPOINT_FORMAT or "cache_fingerprint" not in ckpt:
        raise ValueError("Legacy checkpoint lacks verified dataset identity; start a fresh run with this notebook")
    if ckpt["cache_fingerprint"] != fingerprint:
        raise ValueError("Checkpoint dataset fingerprint differs from the current cache; refusing resume/evaluation")
    if ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint belongs to another model")
    saved_options = saved_architecture(ckpt["args"])
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(saved_options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if expected_model_hash is not None and ckpt.get("model_fingerprint") != expected_model_hash:
        raise ValueError("Checkpoint architecture differs from this notebook")
    if run_id is not None and canonical_run_id(ckpt["args"]["run_name"]) != canonical_run_id(run_id):
        raise ValueError("Checkpoint belongs to another run ID")


def load_model(ckpt, weights, device):
    options = saved_architecture(ckpt["args"])
    model, n_params = build_model(architecture=options)
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if "n_params" in ckpt and ckpt["n_params"] != n_params:
        raise ValueError("Checkpoint parameter count differs from its saved architecture")
    if ckpt.get("model_fingerprint") != model_fingerprint(model) or ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint architecture differs from this notebook")
    model.load_state_dict(ckpt["ema"]["module"] if weights == "ema" else ckpt["model"])
    return model.to(device).eval(), n_params


def validate_submission(path, sample_path):
    sub, sample = pd.read_csv(path), pd.read_csv(sample_path)
    if list(sub.columns) != ["id", "label"] or len(sub) != len(sample):
        raise ValueError("Submission columns or row count differ from the sample")
    if not sub["id"].is_unique or not np.array_equal(sub["id"], sample["id"]):
        raise ValueError("Submission IDs are duplicated or differ from sample order")
    if not pd.api.types.is_integer_dtype(sub["label"]) or not sub["label"].between(0,9).all():
        raise ValueError("Submission labels must be integers in 0..9")
    return len(sub)


def classification_metrics(probs, labels):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    pred = probs.argmax(1)
    cm = np.bincount(labels * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    support = cm.sum(1)
    recall = np.divide(cm.diagonal(), support, out=np.zeros(NUM_CLASSES, dtype=float), where=support > 0)
    return {"accuracy_pct": float((pred == labels).mean()*100), "correct": int((pred == labels).sum()),
            "n": len(labels), "cross_entropy": float(-np.log(probs[np.arange(len(labels)), labels].clip(1e-12)).mean()),
            "per_class_recall": {name: float(value) if count else None
                                 for name, value, count in zip(CLASSES, recall, support)},
            "confusion_matrix": cm.tolist()}

In [23]:
# ---- the training notebooks' exact 45k/5k split, their training-split normalization, and the test images
COMPETITION_CACHE_FINGERPRINT = "42749a486a13184f0f96dab13293950f0120c357ddf1e6b1ef6a3bd3a9ce5fde"
# SHA-256 of the sorted int64 validation indices and their labels (the values Q08_shiftval.ipynb pins). Both equal
# every validation export saved by the A/B/C/E/F/G runs (result/evaluations/validation/<RUN_ID>/*-validation.npz).
EXPECTED_VAL_INDICES_SHA256 = "7957d060fcc34aefa5a43c70e31433927db55526494f67a9f7da553059be6767"
EXPECTED_VAL_LABELS_SHA256 = "aba30157995bda52486927048b705427db4a51e03e0bcb4844d462bb15f20846"
TEST_SIZE = 2000  # the competition's unlabeled test images (assumed 200 per class for balancing)


def int64_sha256(values):
    """SHA-256 of little-endian int64 bytes (indices and labels)."""
    return hashlib.sha256(np.ascontiguousarray(values, dtype="<i8").tobytes()).hexdigest()


if Q10_COMPETITION_CHECKS:
    if META["fingerprint"] != COMPETITION_CACHE_FINGERPRINT:
        raise RuntimeError(f"Cache fingerprint {META['fingerprint'][:12]} is not the competition dataset "
                           f"{COMPETITION_CACHE_FINGERPRINT[:12]}; Q10 must use the training notebooks' data")
    if (VAL_SIZE, SPLIT_SEED) != (5000, 0):
        raise ValueError("Q10 uses the training notebooks' split: keep VAL_SIZE=5000 and SPLIT_SEED=0")
_train_images, _train_labels, _train_ids = load_train(DATA_DIR, CACHE_DIR, mmap=True)
TRAIN_IDX, VAL_IDX = stratified_split(_train_labels, VAL_SIZE, SPLIT_SEED)
if not len(VAL_IDX):
    raise ValueError("Q10 needs a validation split (VAL_SIZE > 0)")
VAL_LABELS_ALL = np.ascontiguousarray(_train_labels[VAL_IDX], dtype=np.int64)
if Q10_COMPETITION_CHECKS:
    if int64_sha256(VAL_IDX) != EXPECTED_VAL_INDICES_SHA256 or int64_sha256(VAL_LABELS_ALL) != EXPECTED_VAL_LABELS_SHA256:
        raise RuntimeError("Validation indices or labels differ from the training notebooks' 5k split")
TRAIN_NORM = channel_stats(_train_images[TRAIN_IDX])  # for comparison with each checkpoint's saved normalization
if type(Q10_SMOKE_N) is not int or Q10_SMOKE_N < 0:
    raise ValueError("Q10_SMOKE_N must be a nonnegative integer")
Q10_N = min(Q10_SMOKE_N, len(VAL_IDX)) if Q10_SMOKE_N else len(VAL_IDX)
Q10_SMOKE = Q10_N != len(VAL_IDX)
VAL_X = np.ascontiguousarray(_train_images[VAL_IDX[:Q10_N]])  # raw uint8 NHWC; normalized per checkpoint
VAL_Y = VAL_LABELS_ALL[:Q10_N]
VAL_X_FULL = VAL_X if not Q10_SMOKE else None  # the Q08 set is generated from the full 5k split only
del _train_images, _train_labels, _train_ids
# Test images in cache order (as predict_run forwards them); TEST_ORDER puts rows into sample-submission order.
TEST_X, TEST_IDS = load_test(DATA_DIR, CACHE_DIR)
SAMPLE_PATH = Path(CACHE_DIR) / "sample_submission.csv"
SAMPLE = pd.read_csv(SAMPLE_PATH)
_row_of = {int(value): i for i, value in enumerate(TEST_IDS)}
if len(_row_of) != len(TEST_IDS) or set(SAMPLE["id"]) != set(_row_of) or len(SAMPLE) != len(TEST_IDS):
    raise ValueError("Test IDs differ from the sample submission or contain duplicates")
if Q10_COMPETITION_CHECKS and len(TEST_IDS) != TEST_SIZE:
    raise ValueError(f"expected {TEST_SIZE:,} test images, found {len(TEST_IDS):,}")
TEST_ORDER = np.array([_row_of[int(value)] for value in SAMPLE["id"]])
SAMPLE_IDS = SAMPLE["id"].to_numpy()
print(f"[split] {len(TRAIN_IDX):,} train / {len(VAL_IDX):,} validation (seed {SPLIT_SEED}); "
      f"indices sha256 {int64_sha256(VAL_IDX)[:12]} | labels sha256 {int64_sha256(VAL_LABELS_ALL)[:12]} | "
      f"cache {META['fingerprint'][:12]}" + (" | competition split verified" if Q10_COMPETITION_CHECKS else ""))
print("Training-split normalization: mean " + ", ".join(f"{v:.6f}" for v in TRAIN_NORM[0])
      + " | std " + ", ".join(f"{v:.6f}" for v in TRAIN_NORM[1]))
print("Validation images per class: " + ", ".join(
    f"{c} {n}" for c, n in zip(CLASSES, np.bincount(VAL_LABELS_ALL, minlength=NUM_CLASSES))))
print(f"[test] {len(TEST_X):,} test images, {len(SAMPLE_IDS):,} unique sample-submission IDs")
if Q10_SMOKE:
    print(f"Warning: Q10_SMOKE_N={Q10_SMOKE_N}: using the first {Q10_N} validation images; results are a smoke "
          "test, not comparable; Q08 and every test file are skipped")

[split] 45,000 train / 5,000 validation (seed 0); indices sha256 7957d060fcc3 | labels sha256 aba30157995b | cache 42749a486a13 | competition split verified
Training-split normalization: mean 0.491325, 0.482039, 0.446408 | std 0.247100, 0.243584, 0.261549
Validation images per class: airplane 500, automobile 500, bird 500, cat 500, deer 500, dog 500, frog 500, horse 500, ship 500, truck 500
[test] 2,000 test images, 2,000 unique sample-submission IDs


## 4. Model: verbatim E code of `E_pyramid_t106_shakedrop.ipynb`

The next two cells are verbatim copies of that notebook's section 4. They rebuild basic PyramidNet E (T106: basic-110,
alpha 98, DropPath 0.1, no ShakeDrop) and the bottleneck ShakeDrop PyramidNets (T122/T128: bottleneck-110 alpha 128;
T123: bottleneck-164 alpha 100; T129: bottleneck-74 alpha 162) from the options saved with each checkpoint, with their
original model fingerprints. In evaluation mode DropPath is the identity, ShakeDrop multiplies each residual branch by
its expectation 1 − p_l and MixStyle is not applied; the BN statistics pass keeps exactly this behaviour.

In [24]:
MODEL = "E"
MODEL_DESCRIPTION = "PyramidNet-110 basic, alpha=98 (widths 16->114)"  # the default (E1/T18, T106) architecture
EXPECTED_PARAMS = 4_963_991  # default basic-110, alpha 98; other BLOCK / DEPTH / ALPHA values: expected_param_count()


class DropPath(nn.Module):
    """Stochastic depth (Huang et al., 2016) for one residual branch, per sample.

    In training, each image's branch is dropped with probability p and survivors are scaled by 1 / (1 - p);
    in evaluation it is the identity. The mask comes from the global torch RNG of the input's device.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.to(x.dtype) / keep

    def extra_repr(self):
        return f"p={self.p:.4f}"


class ShakeDrop(nn.Module):
    """ShakeDrop (Yamada, Iwamura, Akiba & Kise, 2019, arXiv 1802.02375) on one residual branch output F(x).

    p is the probability that this block is perturbed. Training: a gate b ~ Bernoulli(1 - p) per image, then
    alpha ~ U[-1, 1] and beta ~ U[0, 1] per element of F(x) (the paper's pixel-level rule). The forward pass
    multiplies F(x) by b + alpha - b*alpha, the backward pass multiplies the gradient by b + beta - b*beta: a kept
    branch (b = 1) is untouched, a perturbed one is scaled by alpha forward and by beta backward. Evaluation multiplies
    F(x) by E[b + alpha - b*alpha] = 1 - p. Like DropPath, the draws come from the default RNG of the input's device
    (gate, then alpha, then beta), inside the (compiled) training graph; no custom autograd.Function is needed:
    y = beta_coef * F + (alpha_coef - beta_coef) * F.detach() has the forward value alpha_coef * F and the gradient
    beta_coef. Coefficients and the product are computed in float32 and cast back to F's dtype (bf16 under autocast);
    channels_last is preserved. No parameters or buffers.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def coefficients(self, x):
        """(forward, backward) float32 coefficients for a training batch shaped like x (one gate per image).

        b + alpha - b*alpha is computed as b + (1 - b) * alpha, which is exactly 1 for a kept branch (b = 1) and exactly
        alpha for a perturbed one (b = 0); likewise for beta.
        """
        keep = 1.0 - self.p
        gate = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep).to(torch.float32)
        alpha = torch.rand_like(x, dtype=torch.float32) * 2.0 - 1.0  # U[-1, 1]
        beta = torch.rand_like(x, dtype=torch.float32)  # U[0, 1]
        perturbed = 1.0 - gate
        return gate + perturbed * alpha, gate + perturbed * beta

    def forward(self, x):
        if self.p == 0.0:
            return x
        if not self.training:
            return x * (1.0 - self.p)  # E[b + alpha - b*alpha] = keep probability
        forward, backward = self.coefficients(x)
        x32 = x.float()
        return (x32 * backward + x32.detach() * (forward - backward)).to(x.dtype)

    def extra_repr(self):
        return f"p={self.p:.4f}"


def sam_replay_needed(model):
    """True if the training forward draws ShakeDrop coefficients: SAM's second pass then replays the first pass's
    draws by restoring the device RNG (SharpnessAwareMinimization, section 5). DropPath masks stay independent."""
    return any(isinstance(m, ShakeDrop) and m.p > 0 for m in model.modules())


MIXSTYLE_AFTER_STAGES = (1, 2)  # training-only MixStyle (MIXSTYLE, section 1) mixes the outputs of stages 1 and 2


def mix_feature_styles(x, apply, lam, perm, eps):
    """MixStyle (Zhou et al., ICLR 2021), "random" variant, on one training batch; parameter-free.

    Each image's per-channel mean mu and standard deviation sigma over the spatial positions (unbiased variance + eps,
    computed in float32 and detached, as in the reference code) are replaced by lam * stat + (1 - lam) * stat of its
    partner image perm[i]; the result is cast back to x's dtype. `apply` (0-dim bool tensor) selects the mixed batch
    or exactly x; lam has shape (B, 1, 1, 1). The draws come from the trainer's MixStyleSampler (section 5).
    """
    x32 = x.float()
    mu = x32.mean(dim=(2, 3), keepdim=True).detach()
    sigma = (x32.var(dim=(2, 3), keepdim=True) + eps).sqrt().detach()
    mixed_mu = lam * mu + (1 - lam) * mu[perm]
    mixed_sigma = lam * sigma + (1 - lam) * sigma[perm]
    mixed = ((x32 - mu) / sigma * mixed_sigma + mixed_mu).to(x.dtype)
    return torch.where(apply, mixed, x)


class PyramidBlock(nn.Module):
    """BN-conv-BN-ReLU-conv-BN (Han et al., 2017), added to a parameter-free shortcut; no ReLU after the addition.

    The ShakeDrop module is registered only when its rate is positive, so the default block (and the T106 model
    fingerprint, which includes repr(model)) is unchanged.
    """

    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0, shakedrop=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        self.shakedrop = ShakeDrop(shakedrop) if shakedrop else None
        # Shortcut: identity, 2x2 average pool where the block downsamples, zero padding for the extra channels.
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.drop_path(self.bn3(out))
        if self.shakedrop is not None:  # F(x) before the shortcut addition
            out = self.shakedrop(out)
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


class PyramidBottleneck(nn.Module):
    """Bottleneck PyramidNet block (Han et al., 2017; dyhan0920/PyramidNet-PyTorch): BN-conv1x1-BN-ReLU-conv3x3
    (stride)-BN-ReLU-conv1x1(x4)-BN, added to the same parameter-free shortcut (2x2 average pool where the block
    downsamples, zero-padded channels); no ReLU after the addition. `planes` is the pyramid width; the block outputs
    4 * planes channels."""

    expansion = 4

    def __init__(self, in_ch, planes, stride=1, drop_path=0.0, shakedrop=0.0):
        super().__init__()
        out_ch = planes * self.expansion
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, planes, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_ch, 1, bias=False)
        self.bn4 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        self.shakedrop = ShakeDrop(shakedrop) if shakedrop else None
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.conv3(F.relu(self.bn3(out)))
        out = self.drop_path(self.bn4(out))
        if self.shakedrop is not None:
            out = self.shakedrop(out)
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


# BLOCK -> (layers per block, so depth = 3 * layers * n + 2, and the block's output channels per pyramid width)
PYRAMID_BLOCKS = {"basic": (2, 1), "bottleneck": (3, PyramidBottleneck.expansion)}


def blocks_per_stage(depth, block):
    """n blocks per stage: (depth - 2) / 6 for basic blocks, (depth - 2) / 9 for bottlenecks."""
    if block not in PYRAMID_BLOCKS:
        raise ValueError(f"BLOCK = {block!r}; choose one of {tuple(PYRAMID_BLOCKS)}")
    per_stage = 3 * PYRAMID_BLOCKS[block][0]
    if type(depth) is not int or depth < per_stage + 2 or (depth - 2) % per_stage:
        raise ValueError(f"DEPTH = {depth!r} is invalid for {block} blocks: use {per_stage}n + 2 with n >= 1 "
                         f"(e.g. {'110' if block == 'basic' else '164, 200 or 272'})")
    return (depth - 2) // per_stage


class PyramidNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of n pyramidal blocks, BN-ReLU, global average pool, linear.

    n = (depth-2)/6 basic or (depth-2)/9 bottleneck blocks per stage. The width grows by alpha / N after each of the
    N = 3n blocks. As in the reference implementation, the width is accumulated as a float and each block uses
    int(round(width)) channels (bottlenecks output 4x that). Block l of N drops its residual branch with probability
    drop_path * l / N (linear decay rule of Huang et al., 2016), or, with ShakeDrop, is perturbed with probability
    shakedrop * l / N (Yamada et al.'s linear rule with p_L = 1 - shakedrop); the two are exclusive. A training forward
    may receive MixStyle draws for the points after stages MIXSTYLE_AFTER_STAGES; this adds no module, parameter or
    buffer.
    """

    mixstyle_layers = len(MIXSTYLE_AFTER_STAGES)  # number of MixStyle draws a training forward accepts

    def __init__(self, depth=110, alpha=98, drop_path=0.0, num_classes=10, block="basic", shakedrop=0.0):
        super().__init__()
        if drop_path and shakedrop:
            raise ValueError("DropPath and ShakeDrop act on the same residual branch; enable only one of them")
        n = blocks_per_stage(depth, block)
        unit = PyramidBlock if block == "basic" else PyramidBottleneck
        expansion = PYRAMID_BLOCKS[block][1]
        total = 3 * n
        add_rate = alpha / total
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch, width, index = [], 16, 16.0, 0
        for i in range(3):
            layers = []
            for j in range(n):
                index += 1
                width += add_rate
                planes = int(round(width))
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(unit(in_ch, planes, stride, drop_path * index / total, shakedrop * index / total))
                in_ch = planes * expansion
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x, mixstyle=None):
        """`mixstyle` = (apply, lam, perm, eps) from MixStyleSampler.draw, or None. Without draws, and always in
        evaluation mode, this is exactly the plain PyramidNet forward."""
        x = self.stem(x)
        if mixstyle is None or not self.training:
            x = self.stages(x)
        else:
            apply, lam, perm, eps = mixstyle
            layer = 0
            for index, stage in enumerate(self.stages, 1):
                x = stage(x)
                if index in MIXSTYLE_AFTER_STAGES:
                    x = mix_feature_styles(x, apply[layer], lam[layer], perm[layer], eps)
                    layer += 1
        x = F.relu(self.bn(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


def pyramid_widths(depth=110, alpha=98, block="basic"):
    """Pyramid width of every block and output channels of each stage, by PyramidNet's int(round(width)) rule."""
    n = blocks_per_stage(depth, block)
    rate = alpha / (3 * n)
    width, planes = 16.0, []
    for _ in range(3 * n):
        width += rate
        planes.append(int(round(width)))
    expansion = PYRAMID_BLOCKS[block][1]
    return planes, [planes[(i + 1) * n - 1] * expansion for i in range(3)]


# Always-present options (unchanged since E1) and optional ones that are omitted from the saved options when at their
# default, so the E1/T18 and T106 options, descriptions and model fingerprints are reproduced exactly.
ARCH_DEFAULTS = {"implementation_version": 1, "drop_path": 0.0}
ARCH_OPTION_DEFAULTS = {"shakedrop": 0.0, "block": "basic", "depth": 110, "alpha": 98}


def arch_option(options, key):
    return options.get(key, ARCH_OPTION_DEFAULTS[key])


def architecture_from_config():
    return {"drop_path": DROP_PATH, "shakedrop": SHAKEDROP, "block": BLOCK, "depth": DEPTH, "alpha": ALPHA}


def validate_architecture(options):
    """Check the resolved options in place (SHAKEDROP becomes a float, an integral float ALPHA an int)."""
    rate = options["drop_path"]
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not 0 <= rate < 1:
        raise ValueError(f"DROP_PATH = {rate!r}; choose a rate in [0, 1) (0 = no stochastic depth)")
    shake = options["shakedrop"]
    if isinstance(shake, bool) or not isinstance(shake, (int, float)) or not 0 <= shake < 1:
        raise ValueError(f"SHAKEDROP = {shake!r}; choose a rate in [0, 1) (0 = off; 0.5 = the paper's p_L = 0.5)")
    options["shakedrop"] = float(shake)
    if rate > 0 and shake > 0:
        raise ValueError(f"DROP_PATH = {rate!r} and SHAKEDROP = {shake!r}: both act on the same residual branch; "
                         "set one of them to 0")
    blocks_per_stage(options["depth"], options["block"])  # checks BLOCK and DEPTH
    alpha = options["alpha"]
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or not math.isfinite(alpha) or alpha <= 0:
        raise ValueError(f"ALPHA = {alpha!r}; choose a finite positive widening (98 for basic-110)")
    if isinstance(alpha, float) and alpha.is_integer():
        options["alpha"] = int(alpha)  # 100.0 and 100 are the same architecture and fingerprint
    count = expected_param_count(options)
    if count > PARAM_BUDGET:
        raise ValueError(f"{options['block']} PyramidNet-{options['depth']} with alpha {options['alpha']} has "
                         f"{count:,} parameters, over the {PARAM_BUDGET:,} budget")


def describe_architecture(options):
    # The rates are saved with the options and enter the architecture fingerprint: resuming with another rate is
    # rejected, and evaluation rebuilds the saved options whatever the current switches. For the default options the
    # first part is exactly MODEL_DESCRIPTION.
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    planes, ends = pyramid_widths(depth, alpha, block)
    text = (f"PyramidNet-{depth} {block}, alpha={alpha} (widths 16->{planes[-1]}"
            + (f", block outputs {planes[0] * PYRAMID_BLOCKS[block][1]}->{ends[-1]}" if block != "basic" else "")
            + f"), stochastic depth DROP_PATH={float(options['drop_path'])!r}")
    shake = arch_option(options, "shakedrop")
    if shake:
        total = 3 * blocks_per_stage(depth, block)
        text += (f", ShakeDrop SHAKEDROP={float(shake)!r} (block l of L={total} perturbed with probability "
                 f"{float(shake)!r}*l/L, p_L={1 - shake:g}; gate per image, alpha/beta per element)")
    return text


def architecture_feature(options):
    if arch_option(options, "block") != "basic":
        return "bottleneck"
    if arch_option(options, "shakedrop") > 0:
        return "shakedrop"
    return "stochdepth" if options["drop_path"] > 0 else None


def expected_param_count(options):
    """Exact trainable parameters from the widths (bias-free convolutions, BN gamma/beta, linear weight and bias).
    DropPath, ShakeDrop and MixStyle add none."""
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    planes, _ = pyramid_widths(depth, alpha, block)
    expansion = PYRAMID_BLOCKS[block][1]
    total, in_ch = 3 * 16 * 9 + 2 * 16, 16  # stem conv + BN
    for p in planes:
        out = p * expansion
        if block == "basic":
            total += 2 * in_ch + 9 * in_ch * p + 2 * p + 9 * p * p + 2 * p
        else:
            total += 2 * in_ch + in_ch * p + 2 * p + 9 * p * p + 2 * p + p * out + 2 * out
        in_ch = out
    return total + 2 * in_ch + in_ch * NUM_CLASSES + NUM_CLASSES  # final BN + linear


def architecture_topology(options):
    """Stem width, output channels of each stage (the width grows by alpha/N in every block) and blocks per stage."""
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    n = blocks_per_stage(depth, block)
    return {"stem_channels": 16, "stage_channels": pyramid_widths(depth, alpha, block)[1], "blocks": [n, n, n]}


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return PyramidNet(depth=arch_option(options, "depth"), alpha=arch_option(options, "alpha"),
                      drop_path=options["drop_path"], num_classes=num_classes, block=arch_option(options, "block"),
                      shakedrop=arch_option(options, "shakedrop"))

In [25]:
PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission: default-valued optional options (SHAKEDROP 0, basic blocks, depth 110, alpha 98) are dropped,
    # so E1/T18 and T106 options, architecture records and fingerprints are reproduced exactly.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model

Model E: PyramidNet-110 basic, alpha=98 (widths 16->114), stochastic depth DROP_PATH=0.0
Architecture options: {'implementation_version': 1, 'drop_path': 0.0}
Trainable parameters: 4,963,991 (budget 5,000,000)


## 5. Q08 shifted validation: the locked set

The first cell below is a verbatim copy of `E_pyramid_t125_queue.ipynb`'s `q08-set` cell, itself a verbatim copy of
`Q08_shiftval.ipynb`'s corruption definition (13 corruptions × 3 severities of the 5k validation images, version 1) and
fingerprint helpers (the tests check both copies). The second regenerates the whole set (about a minute, ~0.6 GB) and
uses it **only if** its whole-set fingerprint equals the locked `3f12d750b281…` exactly; otherwise Q08 is skipped with
the reason (no Pillow-mismatch exception). Then only the evaluated subset is kept (default: severity 1 of all 13
corruptions, the mildest level and the closest to the measured test shift; `Q08_EVAL_SEVERITIES` /
`Q08_EVAL_CORRUPTIONS`), evaluated for the checkpoints in `Q08_CHECKPOINTS` (default: last, Q08's pre-declared
primary checkpoint). Each set is its own domain: BN statistics are adapted on that set's 5,000 images. Scores per
set, their mean, and Q08's family-balanced score (mean of the photometric, sharpness, noise and compression family means
over the evaluated sets). A diagnostic, not a hidden-test estimate.

In [26]:
# Verbatim copy of Q08_shiftval.ipynb: its corruption-set cell (section 4), then q08_array_sha256, q08_fingerprints,
# q08_set_sha256 and q08_library_versions from its fingerprint cell and int64_sha256 from its split cell.
# Do not edit: tests/test_e_t125_queue.py compares this text with Q08_shiftval.ipynb.
# ---- Q08 corruption set: fixed a priori, version-locked (bump Q08_SET_VERSION for ANY change below)
Q08_SET_VERSION = 1
Q08_SET_NAME = f"Q08_shiftval-v{Q08_SET_VERSION}"
Q08_FAMILIES = ("photometric", "sharpness", "noise", "compression")
CIFAR10C_REFERENCE = {
    "paper": "Hendrycks & Dietterich (2019), Benchmarking Neural Network Robustness to Common Corruptions and Perturbations",
    "repository": "https://github.com/hendrycks/robustness",
    "file": "ImageNet-C/create_c/make_cifar_c.py",
    "file_last_commit": "ca360085d57889288c4143fddd8a707bbcae01bd",
    "file_sha256": "f865ae91dffb2b4834292ce05c876bf3d77f36417f70075a2b5259348b0c83a5",
}
# params[k] is severity k+1. "source" says which values come from CIFAR-10-C (make_cifar_c.py) and which are
# defined here by a stated symmetry. None is fitted to hidden-test statistics or tuned on model accuracy.
Q08_CORRUPTIONS = {
    "brightness_up":   {"family": "photometric", "op": "brightness", "params": [0.05, 0.10, 0.15],
                        "source": "CIFAR-10-C brightness severities 1-3: HSV value V + c, clipped to [0, 1]"},
    "brightness_down": {"family": "photometric", "op": "brightness", "params": [-0.05, -0.10, -0.15],
                        "source": "defined: mirror of CIFAR-10-C brightness (HSV value V - c, same magnitudes)"},
    "contrast_down":   {"family": "photometric", "op": "contrast", "params": [0.75, 0.5, 0.4],
                        "source": "CIFAR-10-C contrast severities 1-3: (x - per-image channel mean) * c + mean"},
    "contrast_up":     {"family": "photometric", "op": "contrast", "params": [1 / 0.75, 1 / 0.5, 1 / 0.4],
                        "source": "defined: reciprocal factors 1/c of CIFAR-10-C contrast (exact inverse before clipping)"},
    "saturate_down":   {"family": "photometric", "op": "saturate", "params": [[1 / 1.5, 0.0], [0.3, 0.0], [0.1, 0.0]],
                        "source": "severity 1 defined as 1/1.5 (inverse of CIFAR-10-C saturate level 3); severities 2-3 = "
                                  "CIFAR-10-C saturate levels 1-2 (HSV S * a + b)"},
    "saturate_up":     {"family": "photometric", "op": "saturate", "params": [[1.5, 0.0], [2.0, 0.1], [2.5, 0.2]],
                        "source": "CIFAR-10-C saturate levels 3-5, its only saturation increases (levels 1-2 desaturate)"},
    "gamma_bright":    {"family": "photometric", "op": "gamma", "params": [0.8625, 0.7370, 0.6215],
                        "source": "defined: x ** g with 0.5 ** g = 0.5 + c for the CIFAR-10-C brightness c = 0.05/0.10/0.15"},
    "gamma_dark":      {"family": "photometric", "op": "gamma", "params": [1.1520, 1.3219, 1.5146],
                        "source": "defined: x ** g with 0.5 ** g = 0.5 - c for the CIFAR-10-C brightness c = 0.05/0.10/0.15"},
    "gaussian_blur":   {"family": "sharpness", "op": "gaussian_blur", "params": [0.4, 0.6, 0.7],
                        "source": "CIFAR-10-C gaussian_blur severities 1-3: sigma (skimage gaussian, nearest edges, truncate 4)"},
    "sharpen":         {"family": "sharpness", "op": "unsharp_mask", "params": [0.4, 0.6, 0.7],
                        "source": "defined: unsharp mask 2x - G_sigma(x) with the CIFAR-10-C blur sigma; to first order the "
                                  "inverse of that blur"},
    "gaussian_noise":  {"family": "noise", "op": "gaussian_noise", "params": [0.04, 0.06, 0.08],
                        "source": "CIFAR-10-C gaussian_noise severities 1-3: additive N(0, c^2) in [0, 1] units"},
    "jpeg":            {"family": "compression", "op": "jpeg", "params": [80, 65, 58],
                        "source": "CIFAR-10-C jpeg_compression severities 1-3: Pillow JPEG quality"},
    "pixelate":        {"family": "compression", "op": "pixelate", "params": [30, 28, 27],
                        "source": "CIFAR-10-C pixelate severities 1-3: int(32 * c) for c = 0.95/0.90/0.85, Pillow BOX down then up"},
}
Q08_SEVERITIES = (1, 2, 3)
# Severity 0 applies each operation's identity parameter (JPEG has none: severity 0 is the clean copy).
Q08_IDENTITY_PARAMS = {"brightness": 0.0, "contrast": 1.0, "saturate": [1.0, 0.0], "gamma": 1.0,
                       "gaussian_blur": 0.0, "unsharp_mask": 0.0, "gaussian_noise": 0.0, "jpeg": None, "pixelate": 32}
Q08_LIBRARY_DEPENDENT_OPS = ("jpeg", "pixelate")  # Pillow/libjpeg code paths; all other operations are NumPy only
Q08_QUANTIZATION = ("float64 arithmetic on x/255; clip to [0, 1]; round half to even (numpy.rint) to uint8. "
                    "CIFAR-10-C truncates with np.uint8 instead (at most one level lower).")
# NumPy-only operations use correctly rounded elementwise arithmetic, exact integer sums, and scalar math/pow
# for the few kernel/curve values, so their bytes do not depend on summation order or SIMD code paths.
Q08_CHUNK = 1000  # images per generation chunk (bounds memory; identical results for any chunk size)
_PIL_BOX = getattr(Image, "Resampling", Image).BOX


def q08_set_key(name, severity):
    return f"{name}__s{severity}"


def q08_set_keys():
    return [q08_set_key(name, s) for name in Q08_CORRUPTIONS for s in Q08_SEVERITIES]


def q08_key_info(key):
    name, severity = key.rsplit("__s", 1)
    spec = Q08_CORRUPTIONS[name]
    return name, int(severity), spec


def q08_library_dependent(key):
    """True for sets made by Pillow/libjpeg code, whose bytes may differ between library builds."""
    name = key.rsplit("__s", 1)[0]
    return name in Q08_CORRUPTIONS and Q08_CORRUPTIONS[name]["op"] in Q08_LIBRARY_DEPENDENT_OPS


def _unit(x_u8):
    return np.asarray(x_u8, dtype=np.float64) / 255.0


def _quantize(x):
    return np.rint(np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)


def op_brightness(x_u8, c):
    """HSV value V -> clip(V + c); hue and saturation fixed (CIFAR-10-C via skimage rgb2hsv/hsv2rgb).

    With H and S fixed, RGB scales by V'/V. Black (V = 0) has S = 0 there, so it becomes grey V'.
    This closed form equals the HSV round trip in exact arithmetic and avoids hue round-off.
    """
    x = _unit(x_u8)
    v = x.max(axis=-1, keepdims=True)
    v_new = np.clip(v + c, 0.0, 1.0)
    scale = np.divide(v_new, v, out=np.zeros_like(v), where=v > 0)
    return _quantize(np.where(v > 0, x * scale, v_new))


def op_contrast(x_u8, c):
    """(x - mean) * c + mean with the per-image, per-channel mean (CIFAR-10-C contrast).

    The mean is an exact integer sum divided once, so it does not depend on NumPy's summation order.
    """
    x_u8 = np.asarray(x_u8)
    x = _unit(x_u8)
    means = x_u8.sum(axis=(1, 2), keepdims=True, dtype=np.int64) / (255.0 * x_u8.shape[1] * x_u8.shape[2])
    return _quantize((x - means) * c + means)


def op_saturate(x_u8, params):
    """HSV saturation S -> clip(S * a + b); hue and value fixed (CIFAR-10-C saturate, closed form).

    A coloured channel is V - V*S*w(H), so it becomes V - (V - x) * S'/S. Grey pixels have S = 0 and hue 0
    (red) in skimage, so S' = b > 0 gives (V, V(1 - b), V(1 - b)); b = 0 leaves them unchanged.
    """
    a, b = params
    x = _unit(x_u8)
    v = x.max(axis=-1, keepdims=True)
    delta = v - x.min(axis=-1, keepdims=True)
    s = np.divide(delta, v, out=np.zeros_like(v), where=delta > 0)
    s_new = np.clip(s * a + b, 0.0, 1.0)
    ratio = np.divide(s_new, s, out=np.zeros_like(s), where=delta > 0)
    coloured = v - (v - x) * ratio
    grey = np.concatenate([v, v * (1.0 - s_new), v * (1.0 - s_new)], axis=-1)
    return _quantize(np.where(delta > 0, coloured, grey))


def op_gamma(x_u8, g):
    """Per-channel power curve x ** g on [0, 1] (g < 1 brightens mid-tones, g > 1 darkens; 0 and 1 fixed)."""
    lut = _quantize(np.array([(k / 255.0) ** g for k in range(256)], dtype=np.float64))
    return lut[np.asarray(x_u8)]


def _gaussian_kernel(sigma, truncate=4.0):
    """scipy.ndimage 1D Gaussian kernel (as used by skimage.filters.gaussian); sigma 0 is the identity."""
    radius = int(truncate * float(sigma) + 0.5) if sigma > 0 else 0
    if radius == 0:
        return np.ones(1)
    # Scalar math.exp: the few weights do not depend on NumPy's vectorized exp implementation.
    weights = np.array([math.exp(-0.5 / (sigma * sigma) * t * t) for t in range(-radius, radius + 1)])
    return weights / weights.sum()


def _blur_hw(x, sigma):
    """Separable Gaussian over height then width of float NHWC images; edges replicate ('nearest')."""
    w = _gaussian_kernel(sigma)
    r = len(w) // 2
    if r == 0:
        return x.copy()
    h, wd = x.shape[1:3]
    p = np.pad(x, ((0, 0), (r, r), (0, 0), (0, 0)), mode="edge")
    out = sum(w[i] * p[:, i:i + h] for i in range(len(w)))
    p = np.pad(out, ((0, 0), (0, 0), (r, r), (0, 0)), mode="edge")
    return sum(w[i] * p[:, :, i:i + wd] for i in range(len(w)))


def op_gaussian_blur(x_u8, sigma):
    """CIFAR-10-C gaussian_blur: skimage gaussian(sigma, multichannel) = per-channel spatial blur."""
    return _quantize(_blur_hw(_unit(x_u8), sigma))


def op_unsharp_mask(x_u8, sigma, amount=1.0):
    """x + amount * (x - G_sigma(x)); amount 1 subtracts the blur's first-order (sigma^2/2) Laplacian term."""
    x = _unit(x_u8)
    return _quantize(x + amount * (x - _blur_hw(x, sigma)))


def op_gaussian_noise(x_u8, sigma, rng):
    """CIFAR-10-C gaussian_noise: x + N(0, sigma^2) per pixel and channel, from this set's own RNG stream."""
    x = _unit(x_u8)
    noise = rng.standard_normal(size=x.shape, dtype=np.float64)  # drawn even for sigma 0: same stream position
    return _quantize(x + sigma * noise)


def op_jpeg(x_u8, quality):
    """CIFAR-10-C jpeg_compression: Pillow JPEG encode at `quality` (default 4:2:0 subsampling) and decode."""
    if quality is None:
        return np.array(x_u8, dtype=np.uint8, copy=True)
    out = np.empty_like(x_u8)
    for i, img in enumerate(x_u8):
        buffer = io.BytesIO()
        Image.fromarray(img).save(buffer, format="JPEG", quality=int(quality))
        buffer.seek(0)
        with Image.open(buffer) as decoded:
            out[i] = np.asarray(decoded.convert("RGB"), dtype=np.uint8)
    return out


def op_pixelate(x_u8, size):
    """CIFAR-10-C pixelate: Pillow BOX resize to size x size, then BOX back to the original size."""
    h, w = x_u8.shape[1:3]
    out = np.empty_like(x_u8)
    for i, img in enumerate(x_u8):
        small = Image.fromarray(img).resize((int(size), int(size)), _PIL_BOX)
        out[i] = np.asarray(small.resize((w, h), _PIL_BOX), dtype=np.uint8)
    return out


Q08_OPS = {"brightness": op_brightness, "contrast": op_contrast, "saturate": op_saturate, "gamma": op_gamma,
           "gaussian_blur": op_gaussian_blur, "unsharp_mask": op_unsharp_mask, "gaussian_noise": op_gaussian_noise,
           "jpeg": op_jpeg, "pixelate": op_pixelate}


def q08_rng(name, severity):
    """Independent stream per corruption x severity: PCG64 seeded by SHA-256("Q08|<name>|<severity>")."""
    digest = hashlib.sha256(f"Q08|{name}|{severity}".encode()).digest()
    return np.random.Generator(np.random.PCG64(np.random.SeedSequence(int.from_bytes(digest[:16], "little"))))


def q08_corrupt(x_u8, name, severity, rng=None, chunk=Q08_CHUNK):
    """Corrupt uint8 NHWC images; images are processed in order, so any chunk size gives the same bytes."""
    x_u8 = np.asarray(x_u8)
    if x_u8.dtype != np.uint8 or x_u8.ndim != 4 or x_u8.shape[-1] != 3:
        raise ValueError("Q08 corruptions take uint8 NHWC RGB images")
    spec = Q08_CORRUPTIONS[name]
    if severity not in (0,) + Q08_SEVERITIES:
        raise ValueError(f"severity must be 0 (identity) or one of {Q08_SEVERITIES}")
    param = Q08_IDENTITY_PARAMS[spec["op"]] if severity == 0 else spec["params"][severity - 1]
    op = Q08_OPS[spec["op"]]
    rng = rng if rng is not None else q08_rng(name, severity)
    out = np.empty_like(x_u8)
    for start in range(0, len(x_u8), max(1, int(chunk))):
        part = x_u8[start:start + chunk]
        out[start:start + chunk] = op(part, param, rng) if spec["op"] == "gaussian_noise" else op(part, param)
    return out


def q08_spec_record():
    """JSON-ready description of the locked set (also hashed into the manifest)."""
    return {"set_name": Q08_SET_NAME, "set_version": Q08_SET_VERSION, "families": list(Q08_FAMILIES),
            "severities": list(Q08_SEVERITIES), "corruptions": Q08_CORRUPTIONS, "quantization": Q08_QUANTIZATION,
            "seeding": "numpy PCG64(SeedSequence(int.from_bytes(sha256('Q08|<name>|<severity>')[:16], 'little')))",
            "cifar10c_reference": CIFAR10C_REFERENCE}


def q08_generate(clean, verbose=True):
    """All corruption x severity copies of `clean` (uint8 NHWC) stacked as (n_sets, N, 32, 32, 3) uint8."""
    keys = q08_set_keys()
    stack = np.empty((len(keys),) + clean.shape, dtype=np.uint8)
    t0 = time.perf_counter()
    for i, key in enumerate(keys):
        name, severity, _ = q08_key_info(key)
        stack[i] = q08_corrupt(clean, name, severity)
    if verbose:
        print(f"[q08] generated {len(keys)} corruption x severity sets of {len(clean):,} images "
              f"in {time.perf_counter() - t0:.1f}s ({stack.nbytes / 2**20:.0f} MiB uint8)")
    return keys, stack


def q08_array_sha256(a):
    """SHA-256 over dtype, shape and C-order bytes, so equal hashes mean bit-identical arrays."""
    a = np.ascontiguousarray(a)
    digest = hashlib.sha256(f"{a.dtype.str}|{'x'.join(str(n) for n in a.shape)}|".encode())
    digest.update(a.tobytes())
    return digest.hexdigest()


def q08_fingerprints(clean, labels, indices, keys, stack):
    hashes = {"clean": q08_array_sha256(clean),
              "labels": q08_array_sha256(np.asarray(labels, dtype="<i8")),
              "val_indices": q08_array_sha256(np.asarray(indices, dtype="<i8"))}
    hashes.update({key: q08_array_sha256(stack[i]) for i, key in enumerate(keys)})
    return hashes


def q08_set_sha256(hashes, version=Q08_SET_VERSION):
    """Whole-set fingerprint: the set version plus every array hash, sorted by name."""
    lines = [f"set_version {version}"] + [f"{key} {hashes[key]}" for key in sorted(hashes)]
    return hashlib.sha256(("\n".join(lines) + "\n").encode()).hexdigest()


def q08_library_versions():
    import PIL
    from PIL import features
    return {"python": platform.python_version(), "numpy": np.__version__, "pillow": PIL.__version__,
            "libjpeg": features.version("jpg"), "libjpeg_turbo": bool(features.check_feature("libjpeg_turbo")),
            "platform": platform.platform()}


def int64_sha256(values):
    """SHA-256 of little-endian int64 bytes (indices and labels)."""
    return hashlib.sha256(np.ascontiguousarray(values, dtype="<i8").tobytes()).hexdigest()

In [27]:
# ---- the locked Q08 set: regenerated from the full 5k validation split, used only if its fingerprint is the lock
Q10_Q08 = {"status": "skipped", "reason": "Q08_EVAL = False", "keys": [], "sets": {}}
if Q08_EVAL and Q10_SMOKE:
    Q10_Q08["reason"] = "smoke subset (Q10_SMOKE_N > 0): the locked set needs all 5,000 validation images"
elif Q08_EVAL:
    _unknown = sorted(set(Q08_EVAL_CORRUPTIONS or ()) - set(Q08_CORRUPTIONS))
    if _unknown or not set(Q08_EVAL_SEVERITIES) <= set(Q08_SEVERITIES) or not Q08_EVAL_SEVERITIES:
        raise ValueError(f"Q08_EVAL_CORRUPTIONS / Q08_EVAL_SEVERITIES must name Q08 corruptions {list(Q08_CORRUPTIONS)} "
                         f"and severities {Q08_SEVERITIES}; unknown {_unknown}")
    _keys, _stack = q08_generate(VAL_X_FULL)
    _hashes = q08_fingerprints(VAL_X_FULL, VAL_Y, VAL_IDX, _keys, _stack)
    _sha = q08_set_sha256(_hashes)
    Q10_Q08.update(set_name=Q08_SET_NAME, set_sha256=_sha, locked_sha256=Q08_LOCKED_SET_SHA256,
                   libraries=q08_library_versions())
    if _sha != Q08_LOCKED_SET_SHA256:
        Q10_Q08["reason"] = (f"regenerated set sha256 {_sha[:12]} differs from the locked {Q08_LOCKED_SET_SHA256[:12]} "
                             f"(libraries {Q10_Q08['libraries']}); Q08 is not used")
        print("!" * 100 + f"\nWARNING: Q08 skipped: {Q10_Q08['reason']}\n" + "!" * 100, flush=True)
    else:
        _names = list(Q08_EVAL_CORRUPTIONS) if Q08_EVAL_CORRUPTIONS else list(Q08_CORRUPTIONS)
        _selected = [q08_set_key(n, s) for n in _names for s in Q08_EVAL_SEVERITIES]
        Q10_Q08.update(status="locked", reason=None, keys=_selected,
                       sets={key: np.ascontiguousarray(_stack[_keys.index(key)]) for key in _selected},
                       families={key: Q08_CORRUPTIONS[q08_key_info(key)[0]]["family"] for key in _selected})
        print(f"[q08] {Q08_SET_NAME}: locked fingerprint {_sha[:12]} reproduced; evaluating {len(_selected)} of "
              f"{len(_keys)} sets (severities {list(Q08_EVAL_SEVERITIES)}): {', '.join(_selected)}", flush=True)
    del _keys, _stack, _hashes
else:
    print("[q08] Q08_EVAL = False: Q08 skipped")

[q08] Q08_EVAL = False: Q08 skipped


## 6. Class-balanced prediction

Verbatim copy of the balancing functions of `Archive/tools/balance_submissions.py` (the tests compare the two). Applied
identically to every domain (validation and Q08 sets: 500 per class; test: assumed 200 per class):

- `sinkhorn_balance`: Sinkhorn iterative proportional fitting of the probability matrix toward uniform class marginals
  (float64, deterministic; stops at a relative marginal error of `BALANCE_TOL`). The balanced probabilities are
  `p_ic w_c` renormalized per image, so the label is the argmax after a per-class threshold shift `log w_c`. For
  label-smoothed models (T106, T126, T129: smoothing 0.1, top probability ~0.91) equal *mass* moves fewer labels and the
  class counts stay further from 200 than for unsmoothed models.
- `exact_balance_labels`: exactly n/k images per class with the maximum total log-probability (linear sum assignment);
  identical for p and for the Sinkhorn-balanced q. Default on validation and test (`BALANCE_EXACT_DOMAINS`).

Reported per variant and view: accuracy, the number of labels balancing changes, and the class-count range.

In [28]:
# Verbatim copy of the balancing functions of Archive/tools/balance_submissions.py (checklist T136). Do not
# edit: tests/test_q10_balance.py compares every definition below with the tool.
# ---- balancing (copied verbatim into Q10_bn_adapt_balance.ipynb; keep the two identical)
BALANCE_VERSION = 1
BALANCE_TOL = 1e-6           # Sinkhorn stops when every class's mass is within this relative error of n / k
BALANCE_MAX_ITER = 10_000
BALANCE_PROB_FLOOR = 1e-12   # probabilities are floored here (float64) so every class keeps some mass


def checked_probabilities(probs):
    """float64 copy of an (n, k) probability matrix with rows renormalized to 1; rejects anything else."""
    p = np.array(probs, dtype=np.float64, copy=True)
    if p.ndim != 2 or p.shape[0] < 1 or p.shape[1] < 2:
        raise ValueError(f"probabilities must be a nonempty (n, k) matrix with k >= 2, got shape {p.shape}")
    if not np.isfinite(p).all() or (p < 0).any():
        raise ValueError("probabilities must be finite and nonnegative")
    rows = p.sum(1)
    if np.abs(rows - 1.0).max() > 1e-3:
        raise ValueError(f"probability rows must sum to 1 (largest deviation {np.abs(rows - 1.0).max():.2e})")
    p = np.maximum(p, BALANCE_PROB_FLOOR)
    return p / p.sum(1, keepdims=True)


def sinkhorn_balance(probs, tol=BALANCE_TOL, max_iter=BALANCE_MAX_ITER):
    """Class-balanced probabilities by Sinkhorn iterative proportional fitting toward uniform class marginals.

    Finds class weights w (k,) such that q_ic = p_ic w_c / sum_d p_id w_d has row sums 1 and column sums n / k (every
    class receives an equal share of the total probability mass). Deterministic float64 iteration: q from the
    current w; stop when max_c |sum_i q_ic / (n / k) - 1| <= tol; else w_c *= (n / k) / sum_i q_ic. The balanced
    label is argmax_c q_ic = argmax_c p_ic w_c (a per-class threshold shift of log w_c). Returns (q, info); w is
    reported scaled to geometric mean 1 (a common factor cancels in q).
    """
    p = checked_probabilities(probs)
    n, k = p.shape
    target = n / k
    w = np.ones(k)
    for iteration in range(1, max_iter + 1):
        q = p * w
        q /= q.sum(1, keepdims=True)
        mass = q.sum(0)
        error = float(np.abs(mass / target - 1.0).max())
        if error <= tol:
            break
        w = w * (target / mass)
    w = w / np.exp(np.log(w).mean())
    info = {"method": "sinkhorn", "balance_version": BALANCE_VERSION, "iterations": iteration,
            "converged": error <= tol, "max_relative_marginal_error": error, "tolerance": tol,
            "target_mass_per_class": target, "class_weights": w.tolist(), "probability_floor": BALANCE_PROB_FLOOR}
    if error > tol:
        raise RuntimeError(f"Sinkhorn balancing did not converge in {max_iter} iterations (error {error:.2e})")
    return q, info


def exact_balance_labels(probs):
    """Exactly n / k images per class with the maximum total log-probability (equal-count optimal assignment).

    Solved exactly as a linear sum assignment (SciPy) of the n images to n slots, n / k slots per class, cost
    -log p_ic. Per-class weights and per-image normalization only shift the total by a constant, so the result
    equals that of the Sinkhorn-balanced probabilities. Requires n divisible by k.
    """
    from scipy.optimize import linear_sum_assignment

    p = checked_probabilities(probs)
    n, k = p.shape
    if n % k:
        raise ValueError(f"exact equal-count assignment needs n divisible by k (n = {n}, k = {k})")
    per_class = n // k
    rows, slots = linear_sum_assignment(np.repeat(-np.log(p), per_class, axis=1))
    labels = np.empty(n, dtype=np.int64)
    labels[rows] = slots // per_class
    return labels


def class_counts(labels, k):
    return np.bincount(np.asarray(labels, dtype=np.int64), minlength=k)


def balance_predictions(probs, exact=True):
    """Argmax, Sinkhorn-balanced and (exact=True) exact equal-count labels of one probability matrix, with counts.

    Returns {"labels": {name: (n,) int64}, "sinkhorn_probabilities": (n, k) float64, "sinkhorn": info,
    "stats": {name: {"changed_vs_argmax", "class_counts", "min_class_count", "max_class_count"}}}.
    """
    p = checked_probabilities(probs)
    k = p.shape[1]
    q, info = sinkhorn_balance(p)
    labels = {"argmax": p.argmax(1).astype(np.int64), "sinkhorn": q.argmax(1).astype(np.int64)}
    if exact:
        labels["exact"] = exact_balance_labels(p)
    stats = {}
    for name, value in labels.items():
        counts = class_counts(value, k)
        stats[name] = {"changed_vs_argmax": int((value != labels["argmax"]).sum()), "class_counts": counts.tolist(),
                       "min_class_count": int(counts.min()), "max_class_count": int(counts.max())}
    return {"labels": labels, "sinkhorn_probabilities": q, "sinkhorn": info, "stats": stats}

## 7. BN adaptation and the evaluation of one checkpoint

The next cell fixes the Q10 protocol (version 1) and defines the evaluation:

- **Statistics** (`bn_target_statistics`): deep copy; BN layers `reset_running_stats()`, `momentum=None`, `train()`;
  everything else `eval()` (checked); equal consecutive batches of at most `BN_STATS_BATCH_SIZE` images in the given
  order (validation index order, Q08 set order, test cache order), normalized with the checkpoint's statistics and
  nothing else; then the parameters are compared bit for bit and each BN layer must have seen every batch.
- **Variants** (`bn_variant_model`): deep copies of the source with blended running statistics; the source model is used
  only for the `source` variant and is compared with its loaded state after every domain and checkpoint.
- **Predictions** (`view_probabilities`): `predict_logits` in eval mode, plain and flip, flip = mean of the two softmax
  outputs; test rows are reordered to the sample submission.
- **Metrics**: accuracy, cross-entropy (none and Sinkhorn), labels changed by balancing, class-count range, and an exact
  McNemar test against the `source` variant with the same view and balancing (b = source wrong & variant right).
- **Checkpoint location and integrity** as in Q05: the run's folders under every root (Drive name, registry name, legacy
  aliases), the checkpoint's SHA-256 against export manifests and prediction reports (a mismatch fails the checkpoint
  unless `Q10_ALLOW_SHA256_MISMATCH`), the training notebooks' `load_checkpoint` / `validate_checkpoint` /
  `load_model`, split indices, parameter count; unfinished runs are skipped.
- **Reproduction**: the `source` probabilities against every stored export of the same checkpoint epoch and weights.
  A mismatch suppresses every test file of the checkpoint; otherwise the reproduced stored test exports are the
  `source` variant's probabilities and unbalanced CSV (not rewritten; listed in the test report), and Q10 adds its
  `-bal` / `-balx` CSVs.
- **Full-data refits** (full-data extension 1, added 2026-11-05; version 1 is unchanged for 45k/5k checkpoints):
  `full_data_split` recognizes a checkpoint saved with `val_size` 0 and an empty `val_idx` (one without the other
  fails) and requires `train_idx` to be all training images; its normalization is compared with the statistics of all
  of them (`full_data_norm`). Clean validation and Q08 are skipped with the reasons in `FULL_DATA_SKIPS` (printed and
  recorded); the reproduction uses the stored test exports only, with the same tolerances, and without a stored export
  of the weights no test file is written; the validation folder gets the per-checkpoint JSON only (no `.npz`).

In [29]:
# ---- Q10 (checklist T136): BN adaptation, evaluation of one checkpoint, reproduction check and outputs.
# Version 1, fixed 2026-11-04 before any Q10 result was seen; a change of the protocol needs Q10_VERSION + 1.
# Full-data extension 1, added 2026-11-05 (after version-1 results of 45k/5k runs, before any full-data checkpoint was
# evaluated): a checkpoint trained on all 50,000 images (VAL_SIZE=0) gets the test domain only (full_data_split). Every
# rule for 45k/5k checkpoints is unchanged, so Q10_VERSION stays 1 and the file names keep "v1".
from fractions import Fraction

Q10_VERSION = 1
Q10_TASK_ID = "T136"
Q10_NAME = f"Q10_bn-adapt-v{Q10_VERSION}"   # shared reports: result/reports/<MM-DD-HH-mm>_Q10_bn-adapt-v1_<role>
Q10_ROLE = f"q10-bnadapt-v{Q10_VERSION}"    # per checkpoint: evaluations/validation/<RUN_ID>/<stamp>_<id>_q10-bnadapt-v1-<tag>
VIEWS = ("plain", "flip")                   # plain = the image; flip = mean softmax of the image and its mirror image
BALANCES = ("none", "sinkhorn", "exact")    # argmax; Sinkhorn-balanced argmax; exact equal-count assignment
FILE_VARIANT = {"source": "bnsrc", "full": "bnfull"}  # test file tags; blends are bnmixNNN (NNN = 100 * alpha)
Q10_FULL_DATA_EXTENSION = 1                 # full-data refits (VAL_SIZE=0): test domain only, test-export baseline
FULL_DATA_SKIPS = {                         # domains of a full-data checkpoint that are skipped, with the reason
    "val": "full-data checkpoint (VAL_SIZE=0): it was trained on these 5,000 validation images",
    "q08": "full-data checkpoint (VAL_SIZE=0): the Q08 sets are corruptions of the 5,000 validation images it was "
           "trained on"}


class Q10Skip(Exception):
    """Expected reason not to evaluate a run or checkpoint: not on this Drive, unsupported, or unfinished training."""


def bn_variant_names(alphas):
    """["source", "full", "mix025", ...]: saved statistics, full replacement, then each blend weight."""
    values = [float(a) for a in alphas]
    if len(set(values)) != len(values) or not all(0.0 < a < 1.0 for a in values):
        raise ValueError(f"blend weights must be distinct values strictly between 0 and 1, got {alphas!r}")
    if any(abs(a * 100 - round(a * 100)) > 1e-9 for a in values):
        raise ValueError(f"blend weights must be multiples of 0.01, got {alphas!r}")
    return ["source", "full"] + [f"mix{round(a * 100):03d}" for a in values]


def bn_variant_alpha(name):
    """Weight of the target statistics: source 0, full 1, mixNNN NNN/100."""
    if name == "source":
        return 0.0
    if name == "full":
        return 1.0
    match = re.fullmatch(r"mix(\d{3})", name)
    if not match:
        raise ValueError(f"unknown BN variant {name!r}")
    return int(match.group(1)) / 100


def file_variant(name):
    return FILE_VARIANT.get(name, f"bn{name}")


def bn_modules(model):
    """(name, module) of every BatchNorm layer, in module order."""
    return [(name, m) for name, m in model.named_modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]


def bn_state(model):
    """CPU clones of every BN buffer (running_mean, running_var, num_batches_tracked): restore checks."""
    return {f"{name}.{key}": value.detach().cpu().clone() for name, m in bn_modules(model)
            for key, value in m.named_buffers(recurse=False)}


def same_state(a, b):
    return a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)


def equal_batches(n, batch_size):
    """Index ranges of ceil(n / batch_size) nearly equal consecutive batches (each at least 2 images)."""
    parts = max(1, math.ceil(n / batch_size))
    bounds = np.linspace(0, n, parts + 1).round().astype(int)
    if n < 2 or np.diff(bounds).min() < 2:
        raise ValueError(f"BN statistics need batches of at least 2 images (n = {n}, batch size {batch_size})")
    return list(zip(bounds[:-1], bounds[1:]))


@torch.no_grad()
def bn_target_statistics(model, images_u8, mean, std, device, batch_size=1000, include_flip=False):
    """BatchNorm running statistics recomputed from `images_u8` (uint8 NHWC; eval transform = normalization only).

    Works on a deep copy, so `model` is never modified. In the copy every BN layer is reset (running mean 0, variance
    1, num_batches_tracked 0), gets momentum=None (cumulative average over batches, as torch.optim.swa_utils.update_bn)
    and is the only module in training mode: the network, ShakeDrop and DropPath stay in evaluation mode (ShakeDrop
    multiplies by 1 - p, DropPath is the identity, MixStyle is never applied), so the forward is the evaluation
    forward except that each BN normalizes with its batch statistics and accumulates them. Equal consecutive batches
    in the given order, then (include_flip) the same batches flipped. No gradients; the copy's parameters are checked
    to be bit-identical afterwards. Returns ({bn name: (running_mean, running_var)} on `device`, info).
    """
    work = copy.deepcopy(model).to(device)
    work.eval()
    layers = bn_modules(work)
    if not layers:
        raise ValueError("the model has no BatchNorm layers")
    momenta = {}
    for name, m in layers:
        if not m.track_running_stats or m.running_mean is None:
            raise ValueError(f"BN layer {name} does not track running statistics")
        momenta[name] = m.momentum
        m.reset_running_stats()
        m.momentum = None
        m.train()
    in_training = [name or "<model>" for name, m in work.named_modules()
                   if m.training and not isinstance(m, nn.modules.batchnorm._BatchNorm)]
    if in_training:
        raise RuntimeError(f"only BN layers may be in training mode for the statistics pass, also: {in_training[:5]}")
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    batches = equal_batches(len(images_u8), batch_size)
    passes = (False, True) if include_flip else (False,)
    for flip in passes:
        for start, stop in batches:
            x = to_float_nchw(torch.as_tensor(np.asarray(images_u8[start:stop])).to(device), mean_t, std_t, False)
            work(x.flip(3) if flip else x)
    for name, m in layers:
        m.momentum = momenta[name]
    work.eval()
    for (name, p), (work_name, q) in zip(model.named_parameters(), work.named_parameters()):
        if name != work_name or not torch.equal(p.detach().to(device), q.detach()):
            raise RuntimeError(f"parameter {name} changed during the BN statistics pass")
    counts = {int(m.num_batches_tracked) for _, m in layers}
    if counts != {len(batches) * len(passes)}:
        raise RuntimeError(f"BN layers saw {sorted(counts)} batches, expected {len(batches) * len(passes)}")
    stats = {name: (m.running_mean.detach().clone(), m.running_var.detach().clone()) for name, m in layers}
    info = {"images": int(len(images_u8)), "include_flip": bool(include_flip), "batches": len(batches) * len(passes),
            "batch_sizes": sorted({int(b - a) for a, b in batches}), "bn_layers": len(layers),
            "momentum": "None (cumulative average of the batch statistics)",
            "modes": "BN layers train (batch statistics), every other module eval; no gradients"}
    del work
    return stats, info


@torch.no_grad()
def bn_variant_model(source, target_stats, alpha):
    """Deep copy of `source` whose BN running statistics are (1 - alpha) * source + alpha * target, means and
    variances blended separately (alpha 1: full replacement; alpha 0: an unchanged copy). Parameters and every other
    buffer are the source's."""
    model = copy.deepcopy(source)
    layers = bn_modules(model)
    if set(target_stats) != {name for name, _ in layers}:
        raise ValueError("target statistics do not match the model's BN layers")
    for name, m in layers:
        t_mean, t_var = target_stats[name]
        if alpha == 1.0:
            m.running_mean.copy_(t_mean)
            m.running_var.copy_(t_var)
        elif alpha != 0.0:
            m.running_mean.mul_(1.0 - alpha).add_(t_mean, alpha=alpha)
            m.running_var.mul_(1.0 - alpha).add_(t_var, alpha=alpha)
    return model.eval()


def view_probabilities(model, images_u8, mean, std, device, batch_size=1000):
    """{"plain", "flip"} float32 softmax probabilities: the training notebooks' val_accuracy path (predict_logits in
    eval mode, fp32, contiguous; flip = 0.5 * (plain + flipped))."""
    plain = F.softmax(predict_logits(model, images_u8, mean, std, device, batch_size), 1).numpy()
    flipped = F.softmax(predict_logits(model, images_u8, mean, std, device, batch_size, flip=True), 1).numpy()
    return {"plain": plain, "flip": 0.5 * (plain + flipped)}


def q10_mcnemar(baseline_correct, new_correct):
    """Paired exact McNemar test (identical to Q05's q05_mcnemar): b = baseline wrong & new right, c = baseline right
    & new wrong; two-sided exact binomial p = min(1, 2 P(X <= min(b, c))), X ~ Bin(b + c, 1/2); p = 1 if b + c = 0."""
    base = np.asarray(baseline_correct, dtype=bool)
    new = np.asarray(new_correct, dtype=bool)
    if base.ndim != 1 or base.shape != new.shape:
        raise ValueError("McNemar needs two equally long one-dimensional correctness vectors")
    b = int(np.count_nonzero(~base & new))
    c = int(np.count_nonzero(base & ~new))
    n = b + c
    tail = sum(math.comb(n, i) for i in range(min(b, c) + 1))
    p = 1.0 if n == 0 else float(min(Fraction(1), Fraction(2 * tail, 2 ** n)))
    return {"b": b, "c": c, "p_value": p}


def balanced_labels(probs, exact):
    """{balance: labels} plus the balancing record (section 6 functions)."""
    out = balance_predictions(probs, exact=exact)
    labels = {"none": out["labels"]["argmax"], "sinkhorn": out["labels"]["sinkhorn"]}
    if exact:
        labels["exact"] = out["labels"]["exact"]
    record = {"sinkhorn": {k: out["sinkhorn"][k] for k in ("iterations", "converged", "max_relative_marginal_error",
                                                            "class_weights")},
              "stats": {("none" if k == "argmax" else k): v for k, v in out["stats"].items()}}
    return labels, out["sinkhorn_probabilities"], record


def evaluate_domain(source, images, labels, norm, alphas, exact, device, keep=False):
    """Every BN variant on one domain (clean validation, one Q08 set, or the test images when labels is None).

    One statistics pass on `images` gives the target statistics; each variant is a deep copy of the source with
    blended statistics (the source model itself is checked to be unchanged afterwards). Per variant and view: softmax
    probabilities, Sinkhorn / exact balanced labels and, with labels, accuracy, cross-entropy and McNemar against the
    source variant with the same view and balancing. Returns (record, arrays): arrays["probs"][variant][view],
    arrays["labels"][variant][view][balance] and (test only) arrays["balanced"][variant][view].
    """
    mean, std = norm
    before = bn_state(source)
    t0 = time.perf_counter()
    target, info = bn_target_statistics(source, images, mean, std, device, BN_STATS_BATCH_SIZE, BN_STATS_INCLUDE_FLIP)
    record = {"n": int(len(images)), "stats_pass": info, "variants": {}}
    arrays = {"probs": {}, "labels": {}, "balanced": {}}
    for name in bn_variant_names(alphas):
        alpha = bn_variant_alpha(name)
        model = source if name == "source" else bn_variant_model(source, target, alpha)
        arrays["probs"][name] = view_probabilities(model, images, mean, std, device, EVAL_BATCH_SIZE)
        if model is not source:
            del model
    if not same_state(before, bn_state(source)):
        raise RuntimeError("the source model's BN statistics changed while evaluating the variants")
    del target
    for name, views in arrays["probs"].items():
        entry = {"alpha": bn_variant_alpha(name), "views": {}}
        arrays["labels"][name], arrays["balanced"][name] = {}, {}
        for view, probs in views.items():
            pred, balanced, balance = balanced_labels(probs, exact)
            arrays["labels"][name][view] = pred
            if labels is None:
                arrays["balanced"][name][view] = balanced
            out = {"balancing": balance}
            if labels is not None:
                ref = arrays["labels"]["source"][view]
                for mode, value in pred.items():
                    correct = value == labels
                    out[mode] = {"accuracy_pct": float(correct.mean() * 100), "correct": int(correct.sum()),
                                 "changed_vs_unbalanced": int((value != pred["none"]).sum()),
                                 "min_class_count": balance["stats"][mode]["min_class_count"],
                                 "max_class_count": balance["stats"][mode]["max_class_count"],
                                 "mcnemar_vs_source": q10_mcnemar(ref[mode] == labels, correct) if name != "source" else None}
                out["none"]["cross_entropy"] = float(-np.log(probs[np.arange(len(labels)), labels].clip(1e-12)).mean())
                out["sinkhorn"]["cross_entropy"] = float(-np.log(balanced[np.arange(len(labels)), labels].clip(1e-12)).mean())
            else:
                ref = arrays["labels"]["source"][view]
                for mode, value in pred.items():
                    out[mode] = {"changed_vs_unbalanced": int((value != pred["none"]).sum()),
                                 "changed_vs_source_same_balance": int((value != ref[mode]).sum()),
                                 "class_counts": balance["stats"][mode]["class_counts"],
                                 "min_class_count": balance["stats"][mode]["min_class_count"],
                                 "max_class_count": balance["stats"][mode]["max_class_count"]}
            entry["views"][view] = out
        record["variants"][name] = entry
    if not keep:
        del arrays["probs"]
    record["seconds"] = round(time.perf_counter() - t0, 2)
    return record, arrays


def q08_aggregate(per_set, families):
    """Mean accuracy over the evaluated Q08 sets and Q08's family-balanced score (mean of the family means)."""
    by_family = {}
    for key, value in per_set.items():
        by_family.setdefault(families[key], []).append(value)
    family = {f: float(np.mean(v)) for f, v in by_family.items()}
    return {"mean_pct": float(np.mean(list(per_set.values()))), "family_pct": family,
            "score_pct": float(np.mean(list(family.values()))), "n_sets": len(per_set)}


# ---- result roots, run folders and recorded artifacts (the Q05 / Q08 search rules)

def _root_key(path):
    return os.path.normcase(os.path.abspath(os.path.expanduser(str(path))))


def wait_for_folder(path, wait_s=0.0):
    """True once `path` is a folder. A Colab MyDrive shortcut can appear a few seconds after mounting: poll briefly."""
    deadline = time.monotonic() + max(0.0, float(wait_s))
    waiting = False
    while True:
        try:
            if path.is_dir():
                return True
        except OSError as error:
            print(f"[q10] cannot access {path} yet: {error}")
        if time.monotonic() >= deadline:
            return False
        if not waiting:
            print(f"[q10] waiting up to {wait_s:g}s for {path} (a Drive shortcut may appear shortly after mounting)",
                  flush=True)
            waiting = True
        time.sleep(min(3.0, max(0.05, deadline - time.monotonic())))


def q10_result_roots():
    """RESULT_DIR, then every available Q10_EXTRA_RESULT_ROOTS folder (read-only). Outputs go to RESULT_DIR only."""
    extras = Q10_EXTRA_RESULT_ROOTS or []
    if not isinstance(extras, (list, tuple)) or not all(isinstance(r, (str, Path)) and str(r) for r in extras):
        raise ValueError("Q10_EXTRA_RESULT_ROOTS must be a list of result folder paths (or empty)")
    own = Path(PATHS.result_dir)
    roots, seen = [own], {_root_key(own)}
    for extra in map(Path, extras):
        if _root_key(extra) in seen:
            continue
        seen.add(_root_key(extra))
        if wait_for_folder(extra, Q10_EXTRA_ROOT_WAIT_S if PATHS.env == "colab" else 0.0):
            roots.append(extra)
            print(f"[q10] extra result root (read-only): {extra}")
        else:
            print(f"[q10] extra result root {extra} not found: runs that exist only there are skipped "
                  "(expected on an account without that shortcut)", flush=True)
    return roots


def root_kind(root):
    return "RESULT_DIR" if _root_key(root) == _root_key(PATHS.result_dir) else "extra (read-only)"


def drive_run_id(entry):
    """Drive folder name of a Q10_RUNS entry (registry names translated with Q10_DRIVE_NAMES)."""
    if not isinstance(entry, str) or Path(entry).name != entry or not re.match(r"^\d{2}-\d{2}-\d{2}-\d{2}_[A-Z]_", entry):
        raise ValueError(f"Q10_RUNS entries are run IDs, got {entry!r}")
    return Q10_DRIVE_NAMES.get(entry, entry)


def run_names(run_id):
    """Folder names to search for a run: its Drive name, its registry name and the training notebooks' aliases."""
    names = [run_id] + [c for c, o in Q10_DRIVE_NAMES.items() if o == run_id]
    names += [legacy for legacy, target in RUN_ID_ALIASES.items() if target == canonical_run_id(run_id)]
    return list(dict.fromkeys(names))


def run_folders(roots, run_id, category):
    """Existing folders of one artifact category (configs, validation, test, manifests, ...) across all roots."""
    found = []
    for root in roots:
        parent = getattr(run_paths(Path(root), run_id), category).parent
        found += [parent / name for name in run_names(run_id) if (parent / name).is_dir()]
    return list(dict.fromkeys(found))


def read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        print(f"Warning: unreadable {path}; ignored")
        return None


def first_json(roots, run_id, category, pattern):
    for folder in run_folders(roots, run_id, category):
        for path in sorted(folder.glob(pattern)):
            data = read_json(path)
            if data:
                return data, path
    return None, None


def report_role(report):
    """best / last of a prediction report in either format (training predict_run or the checkpoint-export notebooks)."""
    if report.get("checkpoint_role") in ("best", "last"):
        return report["checkpoint_role"]
    name = Path(str(report.get("checkpoint") or report.get("checkpoint_file") or "")).name
    return "last" if name.endswith("_last-ckpt.pt") else "best" if name.endswith("_best-ckpt.pt") else None


def prediction_reports(roots, run_id):
    """Every stored test prediction report of the run with its probabilities file (Q10's own files excluded)."""
    found, seen = [], set()
    for folder in run_folders(roots, run_id, "test"):
        for path in sorted(folder.glob("*_prediction-report*.json")):
            if path.name in seen or "_q10-" in path.name:
                continue
            data = read_json(path)
            if not data:
                continue
            seen.add(path.name)
            npz = path.with_name(path.name.replace("_prediction-report", "_probabilities")).with_suffix(".npz")
            found.append({"report": path.name, "folder": str(folder), "role": report_role(data),
                          "epoch": data.get("epoch"), "weights": data.get("weights"), "tta": data.get("tta"),
                          "checkpoint_sha256": data.get("checkpoint_sha256"),
                          "probabilities": npz if npz.is_file() else None})
    return found


def recorded_checkpoint_hashes(roots, run_id, role, reports):
    """SHA-256 values recorded for the run's `role` checkpoint: export manifests and prediction reports."""
    records, seen = [], set()
    suffix = f"_{role}-ckpt.pt"
    for folder in run_folders(roots, run_id, "manifests"):
        for path in sorted(folder.glob("*_manifest*.json")):
            if path.name in seen:
                continue
            seen.add(path.name)
            for item in (read_json(path) or {}).get("files", []):
                name = Path(str(item.get("path", ""))).name
                if str(item.get("path", "")).startswith("checkpoints/") and name.endswith(suffix) and item.get("sha256"):
                    records.append({"source": path.name, "sha256": item["sha256"], "bytes": item.get("bytes")})
    for r in reports:
        if r["role"] == role and r.get("checkpoint_sha256"):
            records.append({"source": r["report"], "sha256": r["checkpoint_sha256"], "bytes": None})
    return records


def integrity(sha256, size, records):
    values = sorted({r["sha256"] for r in records})
    if not values:
        return "unverified: no recorded SHA-256 for this checkpoint"
    if sha256 in values:
        sizes = {r["bytes"] for r in records if r["sha256"] == sha256 and r["bytes"] is not None}
        if sizes and sizes != {size}:
            raise ValueError(f"checkpoint size {size} differs from the recorded {sorted(sizes)} bytes")
        return "verified against " + ", ".join(sorted({r["source"] for r in records if r["sha256"] == sha256}))
    message = f"checkpoint SHA-256 {sha256[:12]} differs from the recorded {[v[:12] for v in values]}"
    if not Q10_ALLOW_SHA256_MISMATCH:
        raise ValueError(message + "; set Q10_ALLOW_SHA256_MISMATCH=True to evaluate it anyway")
    print(f"Warning: {message} (allowed by Q10_ALLOW_SHA256_MISMATCH)")
    return "MISMATCH (allowed): " + message


def locate_checkpoint(run_id, role, roots):
    """(root, folder name, path) of the run's `role` checkpoint in the first root holding it, or None."""
    for root in roots:
        for name in run_names(run_id):
            path = checkpoint_path(Path(root) / "checkpoints" / name, role)
            if path.is_file():
                return Path(root), name, path
    return None


def architecture_supported(options):
    """Canonical options of a saved E architecture this notebook's verbatim E code rebuilds, else Q10Skip."""
    options = dict(ARCH_DEFAULTS) if options is None else options
    if not isinstance(options, dict):
        raise Q10Skip("saved architecture options are not a dictionary")
    unknown = sorted(set(options) - set(ARCH_DEFAULTS) - set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise Q10Skip(f"architecture options {unknown} are not rebuilt by the E code copied here (IBN / head variants)")
    try:
        return resolve_architecture(dict(options))
    except ValueError as error:
        raise Q10Skip(f"invalid architecture options {options}: {error}") from None


def full_data_split(ckpt):
    """None for a checkpoint of the 45k/5k split (its indices are then checked as in version 1); for a full-data refit
    the record of its verified training split. E_pyramid_t106_shakedrop.ipynb saves a VAL_SIZE=0 run with args val_size
    0, an empty val_idx and train_idx = every training image (no TRAIN_SUBSET); an inconsistent record is refused."""
    val_size, n_val = ckpt["args"].get("val_size"), len(ckpt["val_idx"])
    if val_size != 0 and n_val:
        return None
    if val_size != 0 or n_val:
        raise ValueError(f"inconsistent split record: saved val_size {val_size!r} with {n_val:,} validation indices")
    every = np.arange(len(TRAIN_IDX) + len(VAL_IDX))  # all training images of the verified cache
    train_idx = ckpt["train_idx"].numpy()
    if not np.array_equal(train_idx, every) or ckpt.get("n_train", len(every)) != len(every):
        raise ValueError(f"full-data checkpoint: its training indices are not all {len(every):,} training images "
                         f"({len(train_idx):,} saved)")
    return {"extension": Q10_FULL_DATA_EXTENSION, "val_size": 0, "n_train": int(len(train_idx)), "n_val": 0,
            "train_indices_sha256": int64_sha256(train_idx), "domains": ["test"], "skipped": dict(FULL_DATA_SKIPS),
            "baseline": "the stored test probability exports of these weights (no validation export exists); "
                        "without one no test file is written"}


_FULL_DATA_NORM = {}


def full_data_norm():
    """Per-channel mean/std of every training image (what a VAL_SIZE=0 run normalizes with), computed once."""
    if "norm" not in _FULL_DATA_NORM:
        images, _, _ = load_train(DATA_DIR, CACHE_DIR, mmap=True)
        _FULL_DATA_NORM["norm"] = channel_stats(images)
        del images
    return _FULL_DATA_NORM["norm"]


def label_of(run_id):
    match = re.search(r"_([A-Z])_[A-Za-z]+-ver-(\d+)-(T\d+)-", run_id)
    return f"{match.group(3)} ({match.group(1)}{match.group(2)})" if match else run_id


# ---- stored references for the "source" variant (the baseline-reproduction check)

def stored_validation_references(roots, run_id, epoch, weights):
    """[(file, {"plain", "flip"} probabilities)] of the stored validation exports of this epoch and weights."""
    refs, seen = [], set()
    for folder in run_folders(roots, run_id, "validation"):
        for path in sorted(folder.iterdir()):
            if path.name in seen or not re.search(r"_(best|last)-validation(-\d+)?\.json$", path.name):
                continue
            seen.add(path.name)
            data = read_json(path)
            npz = path.with_suffix(".npz")
            if (not data or data.get("epoch") != epoch or data.get("cache_fingerprint") != META["fingerprint"]
                    or not npz.is_file()):
                continue
            with np.load(npz, allow_pickle=False) as z:
                keys = (f"{weights}_no_tta_probabilities", f"{weights}_flip_tta_probabilities")
                if not set(keys) <= set(z.files) or not np.array_equal(z["indices"][:len(VAL_Y)], VAL_IDX[:len(VAL_Y)]):
                    continue
                refs.append((npz.name, {"plain": z[keys[0]][:len(VAL_Y)], "flip": z[keys[1]][:len(VAL_Y)]}))
    return refs


def stored_test_references(reports, sha256, epoch, weights):
    """[(file, view, probabilities in sample order)] of stored test exports of this checkpoint and weights."""
    refs = []
    for r in reports:
        if r["probabilities"] is None or r["weights"] != weights or not isinstance(r["tta"], bool):
            continue
        if r.get("checkpoint_sha256") != sha256 or r["epoch"] != epoch:
            continue
        with np.load(r["probabilities"], allow_pickle=False) as z:
            if not np.array_equal(z["ids"], SAMPLE_IDS):
                continue
            refs.append((Path(r["probabilities"]).name, "flip" if r["tta"] else "plain", z["probabilities"]))
    return refs


def source_test_exports(reports, names):
    """The stored test exports the source variant reproduces (their probabilities files are `names`, the test files of
    the reproduction check): the training-time originals of this checkpoint and weights. Q10 writes no probabilities
    file and no unbalanced CSV for the source variant (they would duplicate these), only its -bal / -balx CSVs."""
    existing = []
    for r in reports:
        npz = r["probabilities"]
        if npz is None or npz.name not in names:
            continue
        csv = npz.with_name(npz.name.replace("_probabilities", "_submission")).with_suffix(".csv")
        existing.append({"view": "flip" if r["tta"] else "plain", "folder": r["folder"], "probabilities": npz.name,
                         "submission": csv.name if csv.is_file() else None, "prediction_report": r["report"]})
    views = {e["view"] for e in existing}
    return {"probabilities_and_unbalanced_csv": "not written: the stored training-time exports listed here",
            "balanced_csvs": "written by Q10 (-bal Sinkhorn, -balx exact)", "existing": existing,
            "views_without_stored_export": [v for v in VIEWS if v not in views]}


def compare_probabilities(reference, reproduced):
    diff = float(np.abs(np.asarray(reference, dtype=np.float64) - reproduced).max())
    changed = int((np.asarray(reference).argmax(1) != np.asarray(reproduced).argmax(1)).sum())
    return {"max_abs_probability_difference": diff, "label_differences": changed,
            "ok": diff <= BASELINE_PROB_ATOL and changed <= BASELINE_MAX_LABEL_CHANGES}


def reproduction(val_refs, val_probs, test_refs, test_probs):
    """The source variant against every stored export; status reproduced / no stored reference / MISMATCH."""
    rows = []
    for name, ref in val_refs:
        for view in VIEWS:
            rows.append(dict(compare_probabilities(ref[view], val_probs[view]), set="validation", view=view, file=name))
    for name, view, ref in test_refs:
        rows.append(dict(compare_probabilities(ref, test_probs[view]), set="test", view=view, file=name))
    status = ("no stored reference" if not rows else "reproduced" if all(r["ok"] for r in rows) else "MISMATCH")
    return {"status": status, "tolerance": {"max_abs_probability_difference": BASELINE_PROB_ATOL,
                                            "label_differences": BASELINE_MAX_LABEL_CHANGES}, "comparisons": rows}


# ---- one checkpoint and weights: clean validation, Q08 sets, test

def inference_settings(device):
    def flag(getter):
        try:
            return bool(getter())
        except Exception:
            return None
    return {"precision": "fp32, no autocast", "memory_format": "contiguous", "batch_size": EVAL_BATCH_SIZE,
            "device": device_name(device), "torch": torch.__version__,
            "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
            "cudnn_allow_tf32": flag(lambda: torch.backends.cudnn.allow_tf32),
            "cuda_matmul_allow_tf32": flag(lambda: torch.backends.cuda.matmul.allow_tf32),
            "flip": "mean of softmax(image) and softmax(horizontal flip), as the training notebooks' val_accuracy",
            "bn_stats_batch_size": BN_STATS_BATCH_SIZE, "bn_stats_include_flip": BN_STATS_INCLUDE_FLIP}


def evaluate_state(model, ckpt, role, weights, roots, run_id, sha256, reports, device, full_data=None):
    """All domains for one loaded model (source statistics = the checkpoint's buffers for these weights); the Q08
    sets only for the roles in Q08_CHECKPOINTS. A full-data checkpoint (`full_data` from full_data_split) was trained
    on the validation images: test domain only, reproduced against the stored test exports alone."""
    norm = (ckpt["norm"]["mean"], ckpt["norm"]["std"])
    exact = set(BALANCE_EXACT_DOMAINS)
    out, arrays = {"domains": {}}, {}
    if full_data is not None:  # full-data extension 1: clean validation and Q08 would measure training images
        out["domains"]["val"] = {"status": "skipped", "reason": FULL_DATA_SKIPS["val"]}
    else:
        val_exact = "val" in exact and len(VAL_X) % NUM_CLASSES == 0  # equal counts need n divisible by 10 (smoke)
        rec, arr = evaluate_domain(model, VAL_X, VAL_Y, norm, BN_BLEND_ALPHAS, val_exact, device, keep=True)
        out["domains"]["val"], arrays["val"] = rec, arr
    q08 = Q10_Q08 if Q10_Q08["status"] == "locked" and role in Q08_CHECKPOINTS and full_data is None else None
    if q08 is not None:
        out["q08"] = {"status": "locked", "set_sha256": q08["set_sha256"], "keys": q08["keys"], "sets": {}}
        arrays["q08"] = {}
        for key in q08["keys"]:
            rec, arr = evaluate_domain(model, q08["sets"][key], VAL_Y, norm, Q08_BLEND_ALPHAS, "q08" in exact, device)
            out["q08"]["sets"][key], arrays["q08"][key] = rec, arr
        out["q08"]["aggregate"] = {}
        for name in bn_variant_names(Q08_BLEND_ALPHAS):
            for view in VIEWS:
                for mode in ("none", "sinkhorn") + (("exact",) if "q08" in exact else ()):
                    per_set = {k: out["q08"]["sets"][k]["variants"][name]["views"][view][mode]["accuracy_pct"]
                               for k in q08["keys"]}
                    out["q08"]["aggregate"][f"{name}|{view}|{mode}"] = q08_aggregate(per_set, q08["families"])
    elif full_data is not None:
        out["q08"] = {"status": "skipped", "reason": FULL_DATA_SKIPS["q08"]}
    elif Q10_Q08["status"] == "locked":
        out["q08"] = {"status": "not evaluated", "reason": f"{role} checkpoint not in Q08_CHECKPOINTS {Q08_CHECKPOINTS}"}
    else:
        out["q08"] = {"status": "skipped", "reason": Q10_Q08.get("reason")}
    test_rec, test_arr = evaluate_domain(model, TEST_X, None, norm, BN_BLEND_ALPHAS, "test" in exact, device, keep=True)
    for name in test_arr["probs"]:  # cache order -> sample-submission order
        for view in VIEWS:
            test_arr["probs"][name][view] = test_arr["probs"][name][view][TEST_ORDER]
            test_arr["balanced"][name][view] = test_arr["balanced"][name][view][TEST_ORDER]
            for mode in test_arr["labels"][name][view]:
                test_arr["labels"][name][view][mode] = test_arr["labels"][name][view][mode][TEST_ORDER]
    out["domains"]["test"], arrays["test"] = test_rec, test_arr
    val_refs = stored_validation_references(roots, run_id, ckpt["epoch"], weights) if full_data is None else []
    test_refs = stored_test_references(reports, sha256, ckpt["epoch"], weights)
    out["reproduction"] = reproduction(val_refs, arrays["val"]["probs"]["source"] if full_data is None else None,
                                       test_refs, test_arr["probs"]["source"])
    out["source_test_exports"] = source_test_exports(reports, {name for name, _, _ in test_refs})
    return out, arrays


def checkpoint_tag(role, epoch, weights):
    return f"{role}-ep{epoch}-{'raw' if weights == 'model' else 'ema'}"


def fresh_path(folder, stem, extension):
    """<stem><ext>, else <stem>-2<ext>, -3, ...: never overwrites."""
    folder.mkdir(parents=True, exist_ok=True)
    path, index = folder / f"{stem}{extension}", 1
    while path.exists():
        index += 1
        path = folder / f"{stem}-{index}{extension}"
    return path


def savez_atomic(path, **arrays):
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("wb") as f:
        np.savez_compressed(f, **arrays)
    replace_file(tmp, path)


def write_test_outputs(state, arrays, artifacts, identity, stamp):
    """Per BN-adapted variant and view: probabilities .npz (ids, probabilities = unbalanced softmax in sample order,
    plus the Sinkhorn-balanced probabilities and labels) and the unbalanced / -bal (Sinkhorn) / -balx (exact) CSVs in
    the existing submission format, each validated against the sample. The source variant writes only its -bal and
    -balx CSVs: its probabilities and unbalanced CSV are the stored training-time export it reproduced (listed in the
    report as source_variant). Then one test report for the checkpoint."""
    tag = state["tag"]
    files = []
    test = arrays["test"]
    for name in test["probs"]:
        source = name == "source"
        for view in VIEWS:
            role = f"q10-{file_variant(name)}-{tag}-{view}"
            entry = {"variant": name, "alpha": bn_variant_alpha(name), "view": view, "probabilities": None,
                     "probabilities_sha256": None, "submissions": {}}
            if not source:
                npz = fresh_path(artifacts.test, f"{stamp}_{identity}_{role}_probabilities", ".npz")
                savez_atomic(npz, ids=SAMPLE_IDS, probabilities=test["probs"][name][view],
                             sinkhorn_probabilities=test["balanced"][name][view].astype(np.float32),
                             **{f"{mode}_labels": value for mode, value in test["labels"][name][view].items()})
                entry.update(probabilities=npz.name, probabilities_sha256=sha256_file(npz))
            stats = state["domains"]["test"]["variants"][name]["views"][view]
            for mode, suffix in (("none", ""), ("sinkhorn", "-bal"), ("exact", "-balx")):
                if mode not in test["labels"][name][view] or (source and mode == "none"):
                    continue
                csv_path = fresh_path(artifacts.test, f"{stamp}_{identity}_{role}{suffix}_submission", ".csv")
                pd.DataFrame({"id": SAMPLE_IDS, "label": test["labels"][name][view][mode].astype(np.int64)}).to_csv(
                    csv_path, index=False)
                rows = validate_submission(csv_path, SAMPLE_PATH)
                if not np.array_equal(pd.read_csv(csv_path)["label"].to_numpy(), test["labels"][name][view][mode]):
                    raise RuntimeError(f"{csv_path.name}: labels differ after writing")
                entry["submissions"][mode] = {"csv": csv_path.name, "sha256": sha256_file(csv_path), "rows": rows,
                                              "format_check": "passed", **stats[mode]}
            files.append(entry)
    report_path = fresh_path(artifacts.test, f"{stamp}_{identity}_{Q10_ROLE}-{tag}_test-report", ".json")
    write_json_atomic({
        "q10": Q10_NAME, "task_id": Q10_TASK_ID, "run_id": state["run_id"], "checkpoint": state["checkpoint"],
        "role": state["role"], "epoch": state["epoch"], "weights": state["weights"], "tag": tag,
        "bn_stats_pass": state["domains"]["test"]["stats_pass"],
        "reproduction": state["reproduction"], "source_variant": state["source_test_exports"],
        "cache_fingerprint": META["fingerprint"], **({"full_data": state["full_data"]} if "full_data" in state else {}),
        "balancing": {"version": BALANCE_VERSION, "sinkhorn_tolerance": BALANCE_TOL,
                      "assumption": "hidden test set balanced, 200 images per class"},
        "files": files, "kaggle_public_accuracy": None,
        "note": ("Generated by Q10_bn_adapt_balance.ipynb (T136); not submitted. Any Kaggle submission needs the "
                 "user's approval; BN adaptation and class balancing must be disclosed in the write-up.")}, report_path)
    return {"report": report_path.name, "files": files}


def write_validation_outputs(state, arrays, artifacts, identity, stamp):
    """Per-checkpoint JSON (every metric) and .npz (float16 validation probabilities of all variants and views;
    predicted labels of every variant, view and balancing on validation and the Q08 sets). A full-data checkpoint has
    neither validation nor Q08 arrays: the JSON only."""
    role = Q10_ROLE + ("-smoke" if Q10_SMOKE else "") + f"-{state['tag']}"
    path = fresh_path(artifacts.validation, f"{stamp}_{identity}_{role}", ".json")
    if "val" not in arrays:  # full-data checkpoint (test domain only)
        state["arrays_file"] = None
        write_json_atomic(state, path)
        return path
    val = arrays["val"]
    names = list(val["probs"])
    payload = {"val_indices": np.asarray(VAL_IDX[:len(VAL_Y)], dtype="<i8"), "val_labels": np.asarray(VAL_Y, dtype="<i8"),
               "variants": np.array(names), "views": np.array(VIEWS),
               "val_probabilities": np.stack([[val["probs"][n][v] for v in VIEWS] for n in names]).astype(np.float16)}
    for mode in BALANCES:
        if mode in val["labels"][names[0]]["plain"]:
            payload[f"val_{mode}_labels"] = np.stack([[val["labels"][n][v][mode] for v in VIEWS] for n in names]).astype(np.uint8)
    if "q08" in arrays:
        keys = list(arrays["q08"])
        q_names = list(arrays["q08"][keys[0]]["labels"])
        payload.update(q08_keys=np.array(keys), q08_variants=np.array(q_names))
        for mode in BALANCES:
            if mode in arrays["q08"][keys[0]]["labels"][q_names[0]]["plain"]:
                payload[f"q08_{mode}_labels"] = np.stack([[[arrays["q08"][k]["labels"][n][v][mode] for v in VIEWS]
                                                           for n in q_names] for k in keys]).astype(np.uint8)
    npz = path.with_suffix(".npz")
    savez_atomic(npz, **payload)
    state["arrays_file"] = npz.name
    write_json_atomic(state, path)
    return path


def summary_rows(state):
    """Long-format rows: one per domain (val, Q08 set, Q08 aggregate, test), variant, view and balancing."""
    base = {"run_id": state["run_id"], "label": state["label"], "checkpoint": state["role"], "epoch": state["epoch"],
            "weights": state["weights"], "reproduction": state["reproduction"]["status"]}
    rows = []

    def add(domain, family, name, view, mode, values):
        mc = values.get("mcnemar_vs_source") or {}
        rows.append(dict(base, domain=domain, family=family, variant=name, alpha=bn_variant_alpha(name), view=view,
                         balance=mode, accuracy_pct=values.get("accuracy_pct"), correct=values.get("correct"),
                         score_pct=values.get("score_pct"), changed_vs_unbalanced=values.get("changed_vs_unbalanced"),
                         changed_vs_source=values.get("changed_vs_source_same_balance"),
                         min_class_count=values.get("min_class_count"), max_class_count=values.get("max_class_count"),
                         mcnemar_b=mc.get("b"), mcnemar_c=mc.get("c"), mcnemar_p=mc.get("p_value")))

    for domain in ("val", "test"):
        for name, entry in state["domains"][domain].get("variants", {}).items():  # none: skipped (full-data)
            for view, out in entry["views"].items():
                for mode in BALANCES:
                    if mode in out:
                        add(domain, None, name, view, mode, out[mode])
    if state["q08"].get("status") == "locked":
        for key, rec in state["q08"]["sets"].items():
            for name, entry in rec["variants"].items():
                for view, out in entry["views"].items():
                    for mode in BALANCES:
                        if mode in out:
                            add(f"q08:{key}", Q10_Q08["families"][key], name, view, mode, out[mode])
        for combo, agg in state["q08"]["aggregate"].items():
            name, view, mode = combo.split("|")
            add("q08:aggregate", "family-balanced", name, view, mode,
                {"accuracy_pct": agg["mean_pct"], "score_pct": agg["score_pct"]})
    return rows


def state_table(state):
    """Compact per-checkpoint table: rows = BN variants; flip view; accuracy % (val, Q08 score), test changes. A
    full-data checkpoint (validation skipped) gets NaN validation columns."""
    rows = []
    q08 = state["q08"] if state["q08"].get("status") == "locked" else None
    val = state["domains"]["val"].get("variants")
    if val is None:  # full-data checkpoint
        nan = {mode: {"accuracy_pct": float("nan")} for mode in BALANCES}
        val = {name: {"views": {view: nan for view in VIEWS}} for name in state["domains"]["test"]["variants"]}
    for name, entry in val.items():
        v = entry["views"]
        t = state["domains"]["test"]["variants"][name]["views"]["flip"]
        row = {"variant": name, "val_plain": v["plain"]["none"]["accuracy_pct"], "val_flip": v["flip"]["none"]["accuracy_pct"],
               "val_flip_bal": v["flip"]["sinkhorn"]["accuracy_pct"],
               "val_flip_balx": v["flip"].get("exact", {}).get("accuracy_pct"),
               "test_flip_changed_vs_src": t["none"]["changed_vs_source_same_balance"],
               "test_flip_bal_changed": t["sinkhorn"]["changed_vs_unbalanced"],
               "test_flip_balx_changed": t.get("exact", {}).get("changed_vs_unbalanced")}
        if q08 is not None and f"{name}|flip|none" in q08["aggregate"]:
            row["q08_flip_score"] = q08["aggregate"][f"{name}|flip|none"]["score_pct"]
            row["q08_flip_bal_score"] = q08["aggregate"][f"{name}|flip|sinkhorn"]["score_pct"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("variant")


def estimate_forwards():
    """Image forwards per checkpoint and weights (statistics pass + plain and flip of every variant): (without Q08,
    with the Q08 sets)."""
    stats_factor = 2 if BN_STATS_INCLUDE_FLIP else 1
    n_clean = len(bn_variant_names(BN_BLEND_ALPHAS))
    n_q08 = len(bn_variant_names(Q08_BLEND_ALPHAS))
    base = (len(VAL_X) + len(TEST_X)) * (stats_factor + 2 * n_clean)
    return base, base + len(Q10_Q08["keys"]) * len(VAL_X) * (stats_factor + 2 * n_q08)


def expected_test_files():
    """(probabilities .npz, CSVs) written per checkpoint and weights: every BN-adapted variant and view a .npz and the
    unbalanced / -bal / -balx CSVs (-balx when "test" is in BALANCE_EXACT_DOMAINS), the source variant per view only
    its -bal / -balx CSVs."""
    balanced = 2 if "test" in BALANCE_EXACT_DOMAINS else 1
    adapted = len(bn_variant_names(BN_BLEND_ALPHAS)) - 1
    return adapted * len(VIEWS), len(VIEWS) * (adapted * (1 + balanced) + balanced)


def evaluate_run(entry, roots, device=None):
    """Every requested checkpoint and weights of one run. Returns ([state, ...], {checkpoint: skip reason})."""
    device = device or DEV
    run_id = drive_run_id(entry)
    config, config_path = first_json(roots, run_id, "configs", "*_config.json")
    if config is not None:
        architecture_supported(config.get("args", {}).get("architecture"))
    status, _ = first_json(roots, run_id, "progress", "*_status.json")
    if status and status.get("status") != "completed":
        raise Q10Skip(f"training not completed (progress {status.get('status')!r} at epoch {status.get('epoch')})")
    located = {role: locate_checkpoint(run_id, role, roots) for role in Q10_CHECKPOINTS}
    missing = {role: "no checkpoint file in " + ", ".join(str(Path(r) / "checkpoints") for r in roots)
               for role, hit in located.items() if hit is None}
    if len(missing) == len(located):
        raise Q10Skip(f"no {'/'.join(Q10_CHECKPOINTS)} checkpoint of this run in any result root "
                      f"({', '.join(str(r) for r in roots)}); it lives on another account's Drive")
    if config is not None and config.get("args", {}).get("val_size") == 0 and "best" in missing:
        missing["best"] = "full-data run (VAL_SIZE=0): training saves no best checkpoint"
    reports = prediction_reports(roots, run_id)
    states = []
    for role, hit in located.items():
        if hit is None:
            print(f"[q10]   {role}: SKIPPED ({missing[role]})", flush=True)
            continue
        root, folder_name, path = hit
        data = path.read_bytes()
        sha256 = hashlib.sha256(data).hexdigest()
        check = integrity(sha256, len(data), recorded_checkpoint_hashes(roots, folder_name, role, reports))
        ckpt = load_checkpoint(io.BytesIO(data))
        del data
        if not isinstance(ckpt, dict) or "args" not in ckpt:
            raise ValueError(f"{path.name} is not a project training checkpoint")
        options = architecture_supported(ckpt["args"].get("architecture"))
        validate_checkpoint(ckpt, META["fingerprint"], run_id=run_id)  # its saved run ID is the Drive name
        full_data = full_data_split(ckpt)  # None: the 45k/5k split, whose indices are checked next as in version 1
        if full_data is None and (not np.array_equal(ckpt["val_idx"].numpy(), VAL_IDX)
                                  or not np.array_equal(ckpt["train_idx"].numpy(), TRAIN_IDX)):
            raise ValueError(f"{path.name}: checkpoint train/validation indices differ from the training notebooks' split")
        if role == "last" and ckpt["epoch"] != ckpt["args"]["epochs"] and not (status and status.get("status") == "completed"):
            missing[role] = f"last checkpoint at epoch {ckpt['epoch']} of {ckpt['args']['epochs']} (unfinished)"
            print(f"[q10]   {role}: SKIPPED ({missing[role]})", flush=True)
            continue
        train_norm = TRAIN_NORM if full_data is None else full_data_norm()  # the statistics of its own training split
        norm_diff = max(abs(a - b) for a, b in zip(list(ckpt["norm"]["mean"]) + list(ckpt["norm"]["std"]),
                                                   list(train_norm[0]) + list(train_norm[1])))
        if full_data is not None:
            print(f"[q10]   {role}: full-data checkpoint, trained on all {full_data['n_train']:,} training images "
                  "(VAL_SIZE=0, split verified): test domain only; baseline = the stored test exports", flush=True)
            for domain, reason in FULL_DATA_SKIPS.items():
                print(f"[q10]   {role}: {'clean validation' if domain == 'val' else 'Q08'} SKIPPED ({reason})", flush=True)
        for weights in Q10_WEIGHTS:
            if weights == "ema" and ckpt.get("ema") is None:
                missing[f"{role}/ema"] = "checkpoint has no EMA weights"
                continue
            t0 = time.perf_counter()
            model, n_params = load_model(ckpt, weights, device)
            if n_params != expected_param_count(options) or n_params > PARAM_BUDGET:
                raise ValueError(f"{path.name}: parameter count {n_params:,} differs from the saved architecture")
            loaded = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            tag = checkpoint_tag(role, ckpt["epoch"], weights)
            print(f"[q10]   {role} epoch {ckpt['epoch']} {weights}: {path.name} from {root_kind(root)}; {check}; "
                  f"{describe_architecture(options)}; {n_params:,} parameters", flush=True)
            state = {"q10": Q10_NAME, "task_id": Q10_TASK_ID, "run_id": folder_name, "label": label_of(folder_name),
                     "role": role, "epoch": ckpt["epoch"], "planned_epochs": ckpt["args"]["epochs"], "weights": weights,
                     "tag": tag, "smoke_subset": Q10_N if Q10_SMOKE else None,
                     "checkpoint": {"file": path.name, "root": str(root), "root_kind": root_kind(root),
                                    "sha256": sha256, "integrity": check,
                                    "config": config_path.name if config_path else None},
                     "architecture": {"options": options, "description": describe_architecture(options),
                                      "n_params": n_params, "model_fingerprint": ckpt.get("model_fingerprint"),
                                      "stochastic_modules": sorted({type(m).__name__ for m in model.modules()
                                                                   if isinstance(m, (ShakeDrop, DropPath))})},
                     "normalization": {"mean": list(ckpt["norm"]["mean"]), "std": list(ckpt["norm"]["std"]),
                                       "max_abs_difference_vs_recomputed_training_split": norm_diff},
                     "bn": {"variants_val_test": bn_variant_names(BN_BLEND_ALPHAS),
                            "variants_q08": bn_variant_names(Q08_BLEND_ALPHAS),
                            "blend": "running_mean / running_var = (1 - a) * source + a * target, separately",
                            "stats_batch_size": BN_STATS_BATCH_SIZE, "stats_include_flip": BN_STATS_INCLUDE_FLIP,
                            "source_statistics": "the checkpoint's buffers of these weights (EMA buffers for EMA)"},
                     "inference": inference_settings(device)}
            if full_data is not None:
                state["full_data"] = full_data
            result, arrays = evaluate_state(model, ckpt, role, weights, roots, folder_name, sha256, reports, device,
                                            full_data)
            state.update(result)
            if not same_state(loaded, {k: v.detach().cpu() for k, v in model.state_dict().items()}):
                raise RuntimeError("the loaded model changed during the evaluation (source statistics not preserved)")
            state["seconds"] = round(time.perf_counter() - t0, 1)
            artifacts = run_paths(Path(PATHS.result_dir), folder_name)
            identity = folder_name.split("_", 1)[1]
            stamp = run_timestamp()
            repro = state["reproduction"]["status"]
            if Q10_SMOKE or not Q10_WRITE_TEST_OUTPUTS:
                state["test_outputs"] = {"status": "not written: " + ("smoke subset" if Q10_SMOKE else "Q10_WRITE_TEST_OUTPUTS = False")}
            elif repro == "MISMATCH":
                state["test_outputs"] = {"status": "not written: source variant did not reproduce the stored exports"}
                print(f"WARNING: {state['label']} {tag}: baseline NOT reproduced: " + "; ".join(
                    f"{r['file']} {r['view']} max diff {r['max_abs_probability_difference']:.2e}, "
                    f"{r['label_differences']} labels" for r in state["reproduction"]["comparisons"] if not r["ok"]))
            elif full_data is not None and repro == "no stored reference":
                state["test_outputs"] = {"status": "not written: full-data checkpoint without a stored test export of "
                                                   "these weights, so the source variant could not be checked"}
                print(f"WARNING: {state['label']} {tag}: no stored test export of these weights to reproduce "
                      "(full-data checkpoint, no validation export); no test file written", flush=True)
            else:
                state["test_outputs"] = dict(status="written", **write_test_outputs(state, arrays, artifacts, identity, stamp))
                missing_views = state["source_test_exports"]["views_without_stored_export"]
                if missing_views:
                    print(f"[q10]   note: {tag} has no stored test export for view(s) {missing_views}, so no original "
                          "(unbalanced source) CSV exists for them; Q10 wrote their source -bal / -balx CSVs", flush=True)
            state["validation_file"] = write_validation_outputs(state, arrays, artifacts, identity, stamp).name
            states.append(state)
            del model, arrays
            if device.type == "cuda":
                torch.cuda.empty_cache()
    return states, missing


def event_path(folder, role, extension):
    """Dated shared Q10 artifact MM-DD-HH-mm_Q10_bn-adapt-v<k>_<role><ext>, numbered on collision."""
    return fresh_path(Path(folder), f"{run_timestamp()}_{Q10_NAME}_{role}", extension)

## 8. Self-test of the BN code (stand-in models)

Before any checkpoint is read, two tiny random PyramidNets with randomized BN buffers (bottleneck depth 11 with ShakeDrop
0.5 and basic depth 8 with DropPath 0.1) check the section 7 code on this device: the source model stays unchanged,
only BN layers are in training mode during the statistics pass, the stem BN statistics equal an independent
computation, the blends are exact (weights 0, 1 and the asymmetric 0.25), the source variant is the training notebooks'
prediction path, `evaluate_domain`'s source / full / blended probabilities equal independent predictions, and balancing
reaches equal class mass. Every reference is computed on the same batches as the code it checks, and the tolerances
depend on the device (`q10_self_test_tolerances`): probabilities within 1e-6 on the CPU, 1e-3 on CUDA / MPS (TF32
convolutions and cuDNN's algorithm choice; up to ~1e-4 between batch sizes was seen on an A100), while the compared
variants must differ by more than 2e-2, so a real error still fails. A failing check stops the notebook here
(`Q10_SELF_TEST = False` skips it).

In [30]:
# ---- stand-in self-test of the BN code on this device (random tiny PyramidNets; no checkpoint is read)
Q10_SELF_TEST_MIN_EFFECT = 2e-2  # BN statistics must move the stand-ins' probabilities by more than this (20x the
                                 # accelerator tolerance), so the tolerances stay far below the effect of a real error


def q10_self_test_tolerances(device):
    """Tolerances of the self-test's floating-point comparisons on `device`. Every reference is computed on the same
    batches as the code it checks. On the CPU the two agree to float32 rounding: probabilities within 1e-6, the stem
    BN statistics within rtol 1e-5 / atol 1e-6 of a float64 recomputation (measured: 1e-7). On CUDA (PyTorch's default
    TF32 convolutions, 10-bit mantissa, rounding 2^-11 ~ 5e-4 per product; cuDNN's per-shape algorithm choice with or
    without cudnn.benchmark) and MPS, equal batches are reproducible in practice but not guaranteed; different batch
    sizes gave up to ~1e-4 in probability on an A100 (torch 2.11, 2026-11-05), so 1e-3 there and rtol 5e-3 / atol 5e-4
    for the statistics. Parameters and buffers are compared bit for bit and blends to float32 rounding on every
    device (copies and elementwise operations, no convolution)."""
    if device.type == "cpu":
        return {"probabilities": 1e-6, "stats_rtol": 1e-5, "stats_atol": 1e-6}
    return {"probabilities": 1e-3, "stats_rtol": 5e-3, "stats_atol": 5e-4}


def q10_self_test(device, tolerances=None):
    """Checks the section 7 BN code on two tiny stand-ins with randomized BN buffers (bottleneck depth 11 with
    ShakeDrop 0.5; basic depth 8 with DropPath 0.1): (1) the given model is unchanged by the statistics pass and the
    variants; (2) during the pass only BN layers are in training mode (forward hooks on the deep copy see ShakeDrop /
    DropPath in eval); (3) the stem BN statistics equal the independently computed average of the batch means and
    unbiased batch variances (for equal batches the mean is the global mean); (4) blend weights 0 / 1 / 0.25 give the
    source / target / 0.75 source + 0.25 target buffers with unchanged parameters (0.25 is asymmetric, so swapped
    weights fail); (5) the source variant's probabilities are the training notebooks' predict_logits path; (6)
    evaluate_domain's source / full / mix025 probabilities equal independent predictions with the source statistics
    and with statistics recomputed on the same batches, and its Sinkhorn-balanced test probabilities have equal class
    mass. References use the same batches as the code they check; the tolerances are q10_self_test_tolerances(device)
    (`tolerances` overrides them, for the tests), and the compared variants must differ by more than
    Q10_SELF_TEST_MIN_EFFECT, i.e. at least 20x the largest probability tolerance."""
    tol = dict(q10_self_test_tolerances(device) if tolerances is None else tolerances)
    worst = [0.0]

    def close(got, want, what):
        diff = float(np.abs(np.asarray(got, dtype=np.float64) - want).max())
        worst[0] = max(worst[0], diff)
        assert diff <= tol["probabilities"], (f"{what}: max probability difference {diff:.2e} > tolerance "
                                              f"{tol['probabilities']:.0e} ({device.type})")

    def moved(a, b, what):
        diff = float(np.abs(np.asarray(a, dtype=np.float64) - b).max())
        assert diff > Q10_SELF_TEST_MIN_EFFECT, (f"{what}: probabilities moved only {diff:.2e}, not more than "
                                                 f"{Q10_SELF_TEST_MIN_EFFECT:.0e}")

    rng = np.random.default_rng(20261104)
    images = rng.integers(0, 256, size=(300, 32, 32, 3), dtype=np.uint8)
    labels = rng.integers(0, NUM_CLASSES, size=300)
    norm = (list(TRAIN_NORM[0]), list(TRAIN_NORM[1]))
    bs = EVAL_BATCH_SIZE  # every reference prediction uses evaluate_domain's batches
    stand_ins = ({"implementation_version": 1, "drop_path": 0.0, "shakedrop": 0.5, "block": "bottleneck", "depth": 11,
                  "alpha": 12},
                 {"implementation_version": 1, "drop_path": 0.1, "depth": 8, "alpha": 12})
    for options in stand_ins:
        worst[0] = 0.0
        torch.manual_seed(0)
        model, _ = build_model(architecture=options)
        with torch.no_grad():
            for _, m in bn_modules(model):
                m.running_mean.normal_(0, 0.5)
                m.running_var.uniform_(0.5, 2.0)
                m.weight.uniform_(0.5, 1.5)
                m.bias.normal_(0, 0.1)
        model = model.to(device).eval()
        before = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        kinds = (ShakeDrop, DropPath, nn.modules.batchnorm._BatchNorm)
        stochastic = [m for m in model.modules() if isinstance(m, (ShakeDrop, DropPath)) and m.p > 0]
        if not stochastic:
            raise RuntimeError("self-test stand-in has no active ShakeDrop / DropPath module")
        # Hooks registered here are deep-copied with the modules, so they fire on bn_target_statistics' copy.
        seen = []
        handles = [m.register_forward_pre_hook(lambda mod, inp: seen.append((isinstance(mod, nn.modules.batchnorm._BatchNorm),
                                                                             mod.training)))
                   for m in model.modules() if isinstance(m, kinds)]
        try:
            stats, info = bn_target_statistics(model, images, *norm, device, batch_size=128)
        finally:
            for handle in handles:
                handle.remove()
        assert info["batch_sizes"] == [100] and info["batches"] == 3, info
        assert {t for is_bn, t in seen if is_bn} == {True} and {t for is_bn, t in seen if not is_bn} == {False}, (
            f"only BN layers may be in training mode during the statistics pass: (is BN, training) seen {set(seen)}")
        assert same_state(before, {k: v.detach().cpu() for k, v in model.state_dict().items()}), "source changed"
        mean_t = torch.tensor(norm[0], device=device).view(1, 3, 1, 1)
        std_t = torch.tensor(norm[1], device=device).view(1, 3, 1, 1)
        parts = equal_batches(300, 128)
        with torch.no_grad():  # the stem convolution on the statistics pass's batches; statistics in float64 on the CPU
            x = to_float_nchw(torch.as_tensor(images).to(device), mean_t, std_t)
            conv = torch.cat([model.stem[0](x[a:b]).cpu() for a, b in parts]).double()
        batch_means = torch.stack([conv[a:b].mean((0, 2, 3)) for a, b in parts])
        batch_vars = torch.stack([conv[a:b].var((0, 2, 3), unbiased=True) for a, b in parts])
        got_mean, got_var = (t.detach().cpu().double() for t in stats["stem.1"])
        for got, want, what in ((got_mean, batch_means.mean(0), "mean"), (got_mean, conv.mean((0, 2, 3)), "global mean"),
                                (got_var, batch_vars.mean(0), "variance")):
            assert torch.allclose(got, want, rtol=tol["stats_rtol"], atol=tol["stats_atol"]), (
                f"stem BN {what} ({device.type}, rtol {tol['stats_rtol']:.0e}): {got} vs {want}")
        zero, one, part = (bn_variant_model(model, stats, a) for a in (0.0, 1.0, 0.25))
        for name, m in bn_modules(model):
            z, o, q = (dict(v.named_modules())[name] for v in (zero, one, part))
            assert torch.equal(z.running_mean, m.running_mean) and torch.equal(z.running_var, m.running_var), (
                f"{name}: blend weight 0")
            assert torch.equal(o.running_mean, stats[name][0]) and torch.equal(o.running_var, stats[name][1]), (
                f"{name}: blend weight 1")
            assert torch.allclose(q.running_mean, 0.75 * m.running_mean + 0.25 * stats[name][0], atol=1e-6), (
                f"{name}: blend weight 0.25 (running_mean)")
            assert torch.allclose(q.running_var, 0.75 * m.running_var + 0.25 * stats[name][1], atol=1e-6), (
                f"{name}: blend weight 0.25 (running_var)")
        assert all(torch.equal(p, q) for p, q in zip(model.parameters(), part.parameters())), "blend changed parameters"
        p_src = view_probabilities(model, images, *norm, device, bs)
        p_zero = view_probabilities(zero, images, *norm, device, bs)
        p_one = view_probabilities(one, images, *norm, device, bs)
        plain = F.softmax(predict_logits(model, images, *norm, device, bs), 1).numpy()
        flipped = F.softmax(predict_logits(model, images, *norm, device, bs, flip=True), 1).numpy()
        for view in VIEWS:
            close(p_zero[view], p_src[view], f"blend weight 0 vs the source ({view})")
        close(p_src["plain"], plain, "source plain vs predict_logits")
        close(p_src["flip"], 0.5 * (plain + flipped), "source flip vs predict_logits")
        moved(p_one["flip"], p_src["flip"], "replacing the randomized BN statistics")
        rec, arr = evaluate_domain(model, images, labels, norm, (0.25,), True, device, keep=True)
        assert list(rec["variants"]) == ["source", "full", "mix025"], list(rec["variants"])
        same_batches, _ = bn_target_statistics(model, images, *norm, device, BN_STATS_BATCH_SIZE, BN_STATS_INCLUDE_FLIP)
        want = {"source": p_src}
        for name, alpha in (("full", 1.0), ("mix025", 0.25)):
            want[name] = view_probabilities(bn_variant_model(model, same_batches, alpha), images, *norm, device, bs)
        for name, ref in want.items():
            for view in VIEWS:
                close(arr["probs"][name][view], ref[view], f"evaluate_domain {name} ({view})")
        for a, b in (("full", "source"), ("mix025", "source"), ("mix025", "full")):
            moved(want[a]["flip"], want[b]["flip"], f"{a} vs {b}")  # a mix-up of any two variants would be caught
        assert rec["variants"]["full"]["views"]["flip"]["exact"]["min_class_count"] == 30
        _, arr_t = evaluate_domain(model, images, None, norm, (0.25,), True, device, keep=True)
        mass = arr_t["balanced"]["full"]["flip"].sum(0)
        assert np.abs(mass / 30 - 1).max() <= 1.001 * BALANCE_TOL, mass
        assert same_state(before, {k: v.detach().cpu() for k, v in model.state_dict().items()}), "source changed"
        print(f"[self-test] {describe_architecture(model._project_architecture)}: passed ({len(bn_modules(model))} BN "
              f"layers in train mode for the pass; {len(stochastic)} {type(stochastic[0]).__name__} modules stayed in "
              f"eval; largest probability difference {worst[0]:.1e}, tolerance {tol['probabilities']:.0e} on "
              f"{device.type})")
        del model, zero, one, part
    if device.type == "cuda":
        torch.cuda.empty_cache()


if Q10_SELF_TEST:
    q10_self_test(DEV)
else:
    print("[self-test] skipped (Q10_SELF_TEST = False)")

[self-test] PyramidNet-11 bottleneck, alpha=12 (widths 16->28, block outputs 80->112), stochastic depth DROP_PATH=0.0, ShakeDrop SHAKEDROP=0.5 (block l of L=3 perturbed with probability 0.5*l/L, p_L=0.5; gate per image, alpha/beta per element): passed (14 BN layers in train mode for the pass; 3 ShakeDrop modules stayed in eval; largest probability difference 0.0e+00, tolerance 1e-03 on cuda)
[self-test] PyramidNet-8 basic, alpha=12 (widths 16->28), stochastic depth DROP_PATH=0.1: passed (11 BN layers in train mode for the pass; 3 DropPath modules stayed in eval; largest probability difference 0.0e+00, tolerance 1e-03 on cuda)


## 9. Evaluate the runs

For each run in `Q10_RUNS`, each saved checkpoint in `Q10_CHECKPOINTS` and each of `Q10_WEIGHTS`: clean validation,
the evaluated Q08 sets (checkpoints in `Q08_CHECKPOINTS`) and the test images, every BN variant × plain/flip × balancing, the reproduction check of the
`source` variant, and the outputs listed at the top. A compact table per checkpoint lists, for the flip view: clean
validation accuracy (none / Sinkhorn / exact), the Q08 family-balanced score (none / Sinkhorn), and how many test labels
change against the source variant and by balancing. The final table summarizes all checkpoints; skipped and failed
runs follow with their reasons. Results are diagnostics on fixed validation sets; any CSV submission needs the user's
approval, and the techniques used must be disclosed in the write-up. Full-data checkpoints (`VAL_SIZE=0`) have test
rows only: their validation and Q08 columns are NaN.

In [31]:
if DEV.type == "cuda":
    torch.backends.cudnn.benchmark = bool(HW.cudnn_benchmark)
Q10_STATES, Q10_SKIPPED, Q10_FAILED = [], {}, {}
_t0 = time.perf_counter()
_roots = q10_result_roots() if Q10_RUNS else []
print("[q10] checkpoint search order: " + " -> ".join(f"{r} ({root_kind(r)})" for r in _roots)
      + f"; outputs go to {PATHS.result_dir} only")
_settings = inference_settings(DEV)
print(f"[q10] inference: {_settings['precision']}, {_settings['memory_format']}, batch {EVAL_BATCH_SIZE}, "
      f"{_settings['device']}; TF32 conv {_settings['cudnn_allow_tf32']} / matmul {_settings['cuda_matmul_allow_tf32']} "
      "(PyTorch defaults, unchanged)")
print(f"[q10] BN variants: validation/test {bn_variant_names(BN_BLEND_ALPHAS)}, Q08 {bn_variant_names(Q08_BLEND_ALPHAS)}; "
      f"statistics from {'un-flipped and flipped' if BN_STATS_INCLUDE_FLIP else 'un-flipped'} images, batches <= "
      f"{BN_STATS_BATCH_SIZE}; balancing: Sinkhorn everywhere, exact on {list(BALANCE_EXACT_DOMAINS)}; Q08 "
      f"{Q10_Q08['status']} ({len(Q10_Q08['keys'])} sets)")
if not set(Q08_CHECKPOINTS) <= {"best", "last"}:
    raise ValueError(f"Q08_CHECKPOINTS = {Q08_CHECKPOINTS!r}; use checkpoint roles best / last")
_plain, _with_q08 = estimate_forwards()
print(f"[q10] image forwards per checkpoint and weights: {_plain / 1e6:.2f}M without Q08, {_with_q08 / 1e6:.2f}M with "
      f"the Q08 sets ({', '.join(Q08_CHECKPOINTS)} checkpoint); at the A100 fp32 rates measured in Q08/Q05 (~8k img/s "
      f"basic-110, ~3.5k bottleneck-110, ~2.5k bottleneck-164) the Q08 checkpoint takes ~{_with_q08 / 8000 / 60:.1f}-"
      f"{_with_q08 / 2500 / 60:.1f} min of forwards, the other ~{_plain / 8000 / 60:.1f}-{_plain / 2500 / 60:.1f} min",
      flush=True)
_npz, _csvs = expected_test_files()
_kinds = "unbalanced/-bal" + ("/-balx" if "test" in BALANCE_EXACT_DOMAINS else "")
print(f"[q10] test files per checkpoint and weights: {_csvs} CSVs and {_npz} probabilities .npz "
      f"({len(bn_variant_names(BN_BLEND_ALPHAS)) - 1} BN-adapted variants x {len(VIEWS)} views x {_kinds}, plus the "
      f"source variant's {_kinds.split('/', 1)[1]} per view) and 1 test report; the source variant's probabilities and "
      "unbalanced CSV are the stored training-time exports (not rewritten)"
      + ("" if Q10_WRITE_TEST_OUTPUTS and not Q10_SMOKE else
         " - none this run (" + ("smoke subset" if Q10_SMOKE else "Q10_WRITE_TEST_OUTPUTS = False") + ")"), flush=True)
for _entry in Q10_RUNS:
    _run = drive_run_id(_entry)
    print(f"\n[q10] {label_of(_run)}: {_run}", flush=True)
    try:
        _states, _missing = evaluate_run(_entry, _roots)
    except Q10Skip as _skip:
        Q10_SKIPPED[_run] = str(_skip)
        print(f"[q10]   SKIPPED: {_skip}", flush=True)
        continue
    except Exception as _error:  # identity, split, integrity or I/O problems: recorded, then the next run
        if Q10_STOP_ON_ERROR:
            raise
        Q10_FAILED[_run] = f"{type(_error).__name__}: {_error}"
        print(f"[q10]   FAILED: {type(_error).__name__}: {_error}", flush=True)
        continue
    for _role, _reason in _missing.items():
        Q10_SKIPPED[f"{_run} {_role}"] = _reason
    for _state in _states:
        Q10_STATES.append(_state)
        _repro = _state["reproduction"]
        print(f"[q10]   {_state['tag']}: baseline {_repro['status']} ("
              + ", ".join(f"{r['set']} {r['view']} max diff {r['max_abs_probability_difference']:.1e} / "
                          f"{r['label_differences']} labels" for r in _repro["comparisons"][:4])
              + (" ..." if len(_repro["comparisons"]) > 4 else "") + f"); {_state['seconds']:.0f}s; test files "
              f"{_state['test_outputs']['status']}; {_state['validation_file']}", flush=True)
        with pd.option_context("display.width", 250, "display.max_columns", None, "display.precision", 2):
            print(state_table(_state).to_string())

Q10_ELAPSED_S = round(time.perf_counter() - _t0, 1)
_role = "summary-smoke" if Q10_SMOKE else "summary"
Q10_SUMMARY_PATH = event_path(Path(PATHS.result_dir) / "reports", _role, ".csv")
Q10_REPORT_PATH = fresh_path(Q10_SUMMARY_PATH.parent, Q10_SUMMARY_PATH.stem.replace(f"_{_role}", f"_{_role.replace('summary', 'report')}"), ".json")
Q10_SUMMARY = pd.DataFrame([row for state in Q10_STATES for row in summary_rows(state)])
Q10_SUMMARY.to_csv(Q10_SUMMARY_PATH, index=False)
write_json_atomic({
    "q10": Q10_NAME, "task_id": Q10_TASK_ID, "version": Q10_VERSION,
    "created_at": datetime.now(ZoneInfo("America/New_York")).isoformat(timespec="seconds"),
    "protocol": {"bn_variants_val_test": bn_variant_names(BN_BLEND_ALPHAS),
                 "bn_variants_q08": bn_variant_names(Q08_BLEND_ALPHAS),
                 "bn_statistics": ("BN layers reset, momentum=None (cumulative average), only BN layers in training "
                                   "mode (ShakeDrop/DropPath eval), eval transform, no gradients, deep copies"),
                 "bn_stats_include_flip": BN_STATS_INCLUDE_FLIP, "bn_stats_batch_size": BN_STATS_BATCH_SIZE,
                 "blend": "(1 - a) * source + a * target for running_mean and running_var separately",
                 "domains": "clean 5k validation (adapted on itself), each Q08 set (adapted on itself), test (adapted on itself)",
                 "balancing": {"sinkhorn_tolerance": BALANCE_TOL, "exact_domains": list(BALANCE_EXACT_DOMAINS),
                               "version": BALANCE_VERSION},
                 "baseline_tolerance": {"max_abs_probability_difference": BASELINE_PROB_ATOL,
                                        "label_differences": BASELINE_MAX_LABEL_CHANGES},
                 "selection": "none: every variant is reported; any Kaggle submission needs the user's approval"},
    "q08": {k: v for k, v in Q10_Q08.items() if k != "sets"}, "inference": _settings,
    "split": {"val_indices_sha256": int64_sha256(VAL_IDX), "n_validation": len(VAL_Y), "smoke_subset": Q10_N if Q10_SMOKE else None,
              "cache_fingerprint": META["fingerprint"]},
    "roots": [{"root": str(r), "kind": root_kind(r)} for r in _roots], "runs": list(Q10_RUNS),
    "evaluated": [f"{s['run_id']} {s['tag']}" for s in Q10_STATES], "skipped": Q10_SKIPPED, "failed": Q10_FAILED,
    "baseline_mismatch": [f"{s['run_id']} {s['tag']}" for s in Q10_STATES if s["reproduction"]["status"] == "MISMATCH"],
    **({"full_data_extension": {"version": Q10_FULL_DATA_EXTENSION, "domains": ["test"], "skipped": FULL_DATA_SKIPS,
                                "checkpoints": [f"{s['run_id']} {s['tag']}" for s in Q10_STATES if "full_data" in s]}}
       if any("full_data" in s for s in Q10_STATES) else {}),
    "elapsed_s": Q10_ELAPSED_S, "summary_csv": Q10_SUMMARY_PATH.name, "states": Q10_STATES}, Q10_REPORT_PATH)
print(f"\n[q10] {len(Q10_STATES)} checkpoint/weights evaluated, {len(Q10_SKIPPED)} skipped, {len(Q10_FAILED)} failed "
      f"in {Q10_ELAPSED_S / 60:.1f} min; summary {Q10_SUMMARY_PATH}; report {Q10_REPORT_PATH.name}")
if len(Q10_SUMMARY):
    _flip = Q10_SUMMARY[Q10_SUMMARY["view"] == "flip"].copy()
    _flip["variant"] = pd.Categorical(_flip["variant"], categories=bn_variant_names(BN_BLEND_ALPHAS), ordered=True)
    _flip["balance"] = pd.Categorical(_flip["balance"], categories=list(BALANCES), ordered=True)
    _key = ["label", "checkpoint", "weights", "variant"]
    _val = _flip[_flip["domain"] == "val"].pivot_table(index=_key, columns="balance", values="accuracy_pct", aggfunc="first",
                                                          observed=True)
    _test = _flip[(_flip["domain"] == "test")].pivot_table(index=_key, columns="balance", values="changed_vs_source",
                                                           aggfunc="first", observed=True)
    _full = _test.index.difference(_val.index)  # full-data checkpoints: test rows only (NaN val_* and q08score_*)
    if len(_full):
        _val = (pd.concat([_val, pd.DataFrame(np.nan, index=_full, columns=_val.columns)]) if len(_val.columns)
                else pd.DataFrame(np.nan, index=_test.index, columns=_test.columns))
    _table = _val.rename(columns=lambda c: f"val_{c}")
    _q08 = _flip[_flip["domain"] == "q08:aggregate"]
    if len(_q08):
        _scores = _q08.pivot_table(index=_key, columns="balance", values="score_pct", aggfunc="first",
                                     observed=True)
        _table = _table.join(_scores.rename(columns=lambda c: f"q08score_{c}"), how="left")
    _table = _table.join(_test.rename(columns=lambda c: f"test_changed_vs_src_{c}"), how="left")
    with pd.option_context("display.width", 250, "display.max_columns", None, "display.precision", 2):
        print(_table.to_string())
    print("Flip view. val_* = clean 5k validation accuracy % (BN adapted on it); q08score_* = Q08's family-balanced "
          "score % over the evaluated sets; test_changed_vs_src_* = test labels that differ from the source variant with "
          "the same balancing; balancing none / sinkhorn / exact. Full long table: the summary CSV.")
    if len(_full):
        print("NaN val_* / q08score_*: full-data checkpoint (VAL_SIZE=0), trained on the validation images: test only.")
for _name, _reason in Q10_SKIPPED.items():
    print(f"SKIPPED {_name}: {_reason}")
for _name, _reason in Q10_FAILED.items():
    print(f"FAILED {_name}: {_reason}")
_mismatch = [f"{s['label']} {s['tag']}" for s in Q10_STATES if s["reproduction"]["status"] == "MISMATCH"]
if _mismatch:
    print(f"WARNING: the source variant did not reproduce the stored exports for {_mismatch}; their test files were "
          "not written (see the report)")

[q10] checkpoint search order: /content/drive/MyDrive/DL_Project1/result (RESULT_DIR); outputs go to /content/drive/MyDrive/DL_Project1/result only
[q10] inference: fp32, no autocast, contiguous, batch 1000, cuda (NVIDIA A100-SXM4-40GB, 39 GB); TF32 conv True / matmul False (PyTorch defaults, unchanged)
[q10] BN variants: validation/test ['source', 'full', 'mix025', 'mix050', 'mix075'], Q08 ['source', 'full', 'mix050']; statistics from un-flipped images, batches <= 1000; balancing: Sinkhorn everywhere, exact on ['val', 'test']; Q08 skipped (0 sets)
[q10] image forwards per checkpoint and weights: 0.08M without Q08, 0.08M with the Q08 sets (last checkpoint); at the A100 fp32 rates measured in Q08/Q05 (~8k img/s basic-110, ~3.5k bottleneck-110, ~2.5k bottleneck-164) the Q08 checkpoint takes ~0.2-0.5 min of forwards, the other ~0.2-0.5 min
[q10] test files per checkpoint and weights: 28 CSVs and 8 probabilities .npz (4 BN-adapted variants x 2 views x unbalanced/-bal/-balx, plus the source